# Stage 4 — Analytical Table Construction

This notebook builds one analytical row per pass rusher and play using only
information available at the exact manual ball snap.

In [2]:
# Stage 4.1 — Configure standalone notebook paths

from pathlib import Path

import pandas as pd


CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = (
    CURRENT_DIR.parent
    if CURRENT_DIR.name == "notebooks"
    else CURRENT_DIR
)

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
INTERIM_DATA_DIR = DATA_DIR / "interim"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

PFF_PATH = RAW_DATA_DIR / "pffScoutingData.csv"
TRACKING_DIR = RAW_DATA_DIR / "tracking_parquets"

PRESSURE_LABELS_PATH = (
    INTERIM_DATA_DIR
    / "pass_rush_pressure_labels.parquet"
)

MANUAL_SNAP_FRAMES_PATH = (
    INTERIM_DATA_DIR
    / "manual_snap_frames.parquet"
)

INPUT_PATHS = {
    "PFF scouting data": PFF_PATH,
    "Tracking directory": TRACKING_DIR,
    "Pressure labels": PRESSURE_LABELS_PATH,
    "Manual snap frames": MANUAL_SNAP_FRAMES_PATH,
}

input_path_validation = pd.DataFrame(
    [
        {
            "input": input_name,
            "relative_path": (
                input_path
                .relative_to(PROJECT_ROOT)
                .as_posix()
            ),
            "exists": input_path.exists(),
        }
        for input_name, input_path in INPUT_PATHS.items()
    ]
)

assert input_path_validation["exists"].all(), (
    "One or more Stage 4 inputs are missing."
)

print(input_path_validation)

                input                                   relative_path  exists
0   PFF scouting data                    data/raw/pffScoutingData.csv    True
1  Tracking directory                      data/raw/tracking_parquets    True
2     Pressure labels  data/interim/pass_rush_pressure_labels.parquet    True
3  Manual snap frames         data/interim/manual_snap_frames.parquet    True


In [4]:
# Stage 4.2 — Load and validate interim data contracts

PRESSURE_LABEL_KEY = [
    "gameId",
    "playId",
    "nflId",
]

SNAP_FRAME_KEY = [
    "gameId",
    "playId",
]

EXPECTED_PRESSURE_LABEL_COLUMNS = [
    *PRESSURE_LABEL_KEY,
    "actual_week",
    "pff_positionLinedUp",
    "pressure",
]

EXPECTED_SNAP_FRAME_COLUMNS = [
    *SNAP_FRAME_KEY,
    "snap_frame_id",
    "actual_week",
]

EXPECTED_PRESSURE_LABEL_ROWS = 36_259
EXPECTED_MANUAL_SNAP_ROWS = 8_532

pressure_labels = pd.read_parquet(
    PRESSURE_LABELS_PATH
)

manual_snap_frames = pd.read_parquet(
    MANUAL_SNAP_FRAMES_PATH
)

assert pressure_labels.columns.tolist() == (
    EXPECTED_PRESSURE_LABEL_COLUMNS
)

assert manual_snap_frames.columns.tolist() == (
    EXPECTED_SNAP_FRAME_COLUMNS
)

assert len(pressure_labels) == EXPECTED_PRESSURE_LABEL_ROWS
assert len(manual_snap_frames) == EXPECTED_MANUAL_SNAP_ROWS

assert not pressure_labels.duplicated(
    subset=PRESSURE_LABEL_KEY
).any()

assert not manual_snap_frames.duplicated(
    subset=SNAP_FRAME_KEY
).any()

assert pressure_labels.isna().sum().sum() == 0
assert manual_snap_frames.isna().sum().sum() == 0

interim_input_summary = pd.DataFrame(
    [
        {
            "input": "pressure_labels",
            "rows": len(pressure_labels),
            "columns": pressure_labels.shape[1],
            "duplicated_keys": int(
                pressure_labels.duplicated(
                    subset=PRESSURE_LABEL_KEY
                ).sum()
            ),
            "missing_values": int(
                pressure_labels.isna().sum().sum()
            ),
        },
        {
            "input": "manual_snap_frames",
            "rows": len(manual_snap_frames),
            "columns": manual_snap_frames.shape[1],
            "duplicated_keys": int(
                manual_snap_frames.duplicated(
                    subset=SNAP_FRAME_KEY
                ).sum()
            ),
            "missing_values": int(
                manual_snap_frames.isna().sum().sum()
            ),
        },
    ]
)

print(interim_input_summary)

                input   rows  columns  duplicated_keys  missing_values
0     pressure_labels  36259        6                0               0
1  manual_snap_frames   8532        4                0               0


In [6]:
# Stage 4.3 — Inventory physical tracking Parquet sources

BYTES_PER_MIB = 1024**2
EXPECTED_TRACKING_FILE_COUNT = 8

EXPECTED_TRACKING_FILE_NAMES = {
    f"week{week_number}.parquet"
    for week_number in range(1, 9)
}

discovered_tracking_paths = list(
    TRACKING_DIR.glob("*.parquet")
)

observed_tracking_file_names = {
    path.name
    for path in discovered_tracking_paths
}

assert (
    len(discovered_tracking_paths)
    == EXPECTED_TRACKING_FILE_COUNT
)

assert (
    observed_tracking_file_names
    == EXPECTED_TRACKING_FILE_NAMES
)

assert all(
    path.stat().st_size > 0
    for path in discovered_tracking_paths
)

tracking_paths = sorted(
    discovered_tracking_paths,
    key=lambda path: int(
        path.stem.removeprefix("week")
    ),
)

tracking_file_inventory = pd.DataFrame(
    [
        {
            "source_file": path.name,
            "source_number": int(
                path.stem.removeprefix("week")
            ),
            "size_mib": round(
                path.stat().st_size / BYTES_PER_MIB,
                3,
            ),
        }
        for path in tracking_paths
    ]
)

print(tracking_file_inventory)

     source_file  source_number  size_mib
0  week1.parquet              1    31.337
1  week2.parquet              2    29.244
2  week3.parquet              3    15.657
3  week4.parquet              4    15.784
4  week5.parquet              5    30.165
5  week6.parquet              6    30.763
6  week7.parquet              7    27.309
7  week8.parquet              8    52.782


In [8]:
# Stage 4.4 — Load minimal PFF role assignments

PFF_ROLE_COLUMNS = [
    "gameId",
    "playId",
    "nflId",
    "pff_role",
]

PFF_PLAYER_PLAY_KEY = [
    "gameId",
    "playId",
    "nflId",
]

EXPECTED_PFF_ROWS = 188_254

EXPECTED_PFF_ROLES = {
    "Coverage",
    "Pass",
    "Pass Block",
    "Pass Route",
    "Pass Rush",
}

pff_roles = pd.read_csv(
    PFF_PATH,
    usecols=PFF_ROLE_COLUMNS,
)

assert len(pff_roles) == EXPECTED_PFF_ROWS

assert pff_roles.columns.tolist() == PFF_ROLE_COLUMNS

assert not pff_roles.duplicated(
    subset=PFF_PLAYER_PLAY_KEY
).any()

assert pff_roles[PFF_ROLE_COLUMNS].notna().all().all()

assert set(pff_roles["pff_role"].unique()) == (
    EXPECTED_PFF_ROLES
)

pff_role_summary = (
    pff_roles["pff_role"]
    .value_counts()
    .rename_axis("pff_role")
    .reset_index(name="rows")
)

pff_role_summary["percentage"] = (
    pff_role_summary["rows"]
    .div(len(pff_roles))
    .mul(100)
    .round(2)
)

print(pff_role_summary)

     pff_role   rows  percentage
0    Coverage  57765       30.68
1  Pass Block  46057       24.47
2  Pass Route  39513       20.99
3   Pass Rush  36362       19.32
4        Pass   8557        4.55


In [10]:
# Stage 4.5 — Build the modeling play index

EXPECTED_MODELING_PLAY_COUNT = 8_531

label_play_index = (
    pressure_labels[
        ["gameId", "playId", "actual_week"]
    ]
    .drop_duplicates()
)

modeling_play_index = label_play_index.merge(
    manual_snap_frames,
    on=[
        "gameId",
        "playId",
        "actual_week",
    ],
    how="left",
    validate="one_to_one",
    indicator=True,
)

assert len(modeling_play_index) == (
    EXPECTED_MODELING_PLAY_COUNT
)

assert modeling_play_index["_merge"].eq("both").all()

assert not modeling_play_index.duplicated(
    subset=SNAP_FRAME_KEY
).any()

assert modeling_play_index[
    "snap_frame_id"
].notna().all()

assert modeling_play_index[
    "snap_frame_id"
].gt(0).all()

modeling_play_index = (
    modeling_play_index
    .drop(columns="_merge")
    .sort_values(SNAP_FRAME_KEY)
    .reset_index(drop=True)
)

modeling_play_summary = (
    modeling_play_index
    .groupby("actual_week", as_index=False)
    .agg(
        modeling_plays=("playId", "size"),
    )
)

pass_rush_rows_by_week = (
    pressure_labels
    .groupby("actual_week", as_index=False)
    .size()
    .rename(columns={"size": "pass_rush_rows"})
)

modeling_play_summary = modeling_play_summary.merge(
    pass_rush_rows_by_week,
    on="actual_week",
    how="left",
    validate="one_to_one",
)

assert modeling_play_summary["modeling_plays"].sum() == 8_531
assert modeling_play_summary["pass_rush_rows"].sum() == 36_259

print(modeling_play_summary)

   actual_week  modeling_plays  pass_rush_rows
0            1            1172            4988
1            2            1062            4498
2            3            1139            4809
3            4            1108            4733
4            5            1105            4681
5            6            1001            4241
6            7             913            3865
7            8            1031            4444


In [11]:
# Stage 4.6 — Restrict PFF roles to modeling plays

EXPECTED_PLAYERS_PER_PLAY = 22

EXPECTED_MODELING_PLAYER_PLAY_ROWS = (
    EXPECTED_MODELING_PLAY_COUNT
    * EXPECTED_PLAYERS_PER_PLAY
)

modeling_pff_roles = (
    modeling_play_index[
        ["gameId", "playId", "actual_week"]
    ]
    .merge(
        pff_roles,
        on=SNAP_FRAME_KEY,
        how="left",
        validate="one_to_many",
    )
)

assert len(modeling_pff_roles) == (
    EXPECTED_MODELING_PLAYER_PLAY_ROWS
)

assert modeling_pff_roles[
    [
        *PFF_PLAYER_PLAY_KEY,
        "pff_role",
        "actual_week",
    ]
].notna().all().all()

players_per_modeling_play = (
    modeling_pff_roles
    .groupby(SNAP_FRAME_KEY)
    .size()
)

assert players_per_modeling_play.eq(
    EXPECTED_PLAYERS_PER_PLAY
).all()

passers_per_modeling_play = (
    modeling_pff_roles.loc[
        modeling_pff_roles["pff_role"].eq("Pass")
    ]
    .groupby(SNAP_FRAME_KEY)
    .size()
)

assert len(passers_per_modeling_play) == (
    EXPECTED_MODELING_PLAY_COUNT
)

assert passers_per_modeling_play.eq(1).all()

modeling_pff_role_summary = (
    modeling_pff_roles["pff_role"]
    .value_counts()
    .rename_axis("pff_role")
    .reset_index(name="rows")
)

print(modeling_pff_role_summary)

     pff_role   rows
0    Coverage  57582
1  Pass Block  45921
2  Pass Route  39389
3   Pass Rush  36259
4        Pass   8531


In [12]:
# Stage 4.7 — Build validated player-role indexes

quarterback_index = (
    modeling_pff_roles.loc[
        modeling_pff_roles["pff_role"].eq("Pass"),
        [
            *PFF_PLAYER_PLAY_KEY,
            "actual_week",
        ],
    ]
    .sort_values(PFF_PLAYER_PLAY_KEY)
    .reset_index(drop=True)
)

pass_blocker_index = (
    modeling_pff_roles.loc[
        modeling_pff_roles["pff_role"].eq("Pass Block"),
        [
            *PFF_PLAYER_PLAY_KEY,
            "actual_week",
        ],
    ]
    .sort_values(PFF_PLAYER_PLAY_KEY)
    .reset_index(drop=True)
)

pass_rusher_index = (
    pressure_labels
    .sort_values(PRESSURE_LABEL_KEY)
    .reset_index(drop=True)
)

pff_pass_rush_keys = modeling_pff_roles.loc[
    modeling_pff_roles["pff_role"].eq("Pass Rush"),
    PFF_PLAYER_PLAY_KEY,
]

pass_rush_key_comparison = pff_pass_rush_keys.merge(
    pass_rusher_index[PFF_PLAYER_PLAY_KEY],
    on=PFF_PLAYER_PLAY_KEY,
    how="outer",
    indicator=True,
    validate="one_to_one",
)

assert pass_rush_key_comparison["_merge"].eq("both").all()

assert len(quarterback_index) == EXPECTED_MODELING_PLAY_COUNT
assert len(pass_rusher_index) == EXPECTED_PRESSURE_LABEL_ROWS

assert (
    pass_blocker_index[SNAP_FRAME_KEY]
    .drop_duplicates()
    .shape[0]
    == EXPECTED_MODELING_PLAY_COUNT
)

role_index_summary = pd.DataFrame(
    [
        {
            "entity_group": "quarterbacks",
            "player_play_rows": len(quarterback_index),
            "plays": (
                quarterback_index[SNAP_FRAME_KEY]
                .drop_duplicates()
                .shape[0]
            ),
        },
        {
            "entity_group": "pass_blockers",
            "player_play_rows": len(pass_blocker_index),
            "plays": (
                pass_blocker_index[SNAP_FRAME_KEY]
                .drop_duplicates()
                .shape[0]
            ),
        },
        {
            "entity_group": "pass_rushers",
            "player_play_rows": len(pass_rusher_index),
            "plays": (
                pass_rusher_index[SNAP_FRAME_KEY]
                .drop_duplicates()
                .shape[0]
            ),
        },
    ]
)

print(role_index_summary)

    entity_group  player_play_rows  plays
0   quarterbacks              8531   8531
1  pass_blockers             45921   8531
2   pass_rushers             36259   8531


In [14]:
# Stage 4.8 — Extract exact manual-snap tracking frames

import pyarrow.parquet as pq


TRACKING_BATCH_SIZE = 250_000
EXPECTED_ENTITIES_PER_SNAP = 23

EXPECTED_SNAP_TRACKING_ROWS = (
    EXPECTED_MODELING_PLAY_COUNT
    * EXPECTED_ENTITIES_PER_SNAP
)

TRACKING_COLUMNS = [
    "gameId",
    "playId",
    "nflId",
    "frameId",
    "time",
    "jerseyNumber",
    "team",
    "playDirection",
    "x",
    "y",
    "s",
    "a",
    "dis",
    "o",
    "dir",
    "event",
]

snap_lookup = modeling_play_index[
    [
        "gameId",
        "playId",
        "snap_frame_id",
        "actual_week",
    ]
]

snap_tracking_batches = []

for tracking_path in tracking_paths:
    parquet_file = pq.ParquetFile(tracking_path)

    for record_batch in parquet_file.iter_batches(
        batch_size=TRACKING_BATCH_SIZE,
        columns=TRACKING_COLUMNS,
    ):
        batch_data = record_batch.to_pandas()

        batch_matches = batch_data.merge(
            snap_lookup,
            on=SNAP_FRAME_KEY,
            how="inner",
            validate="many_to_one",
        )

        batch_snap = batch_matches.loc[
            batch_matches["frameId"].eq(
                batch_matches["snap_frame_id"]
            )
        ].copy()

        if not batch_snap.empty:
            batch_snap["source_file"] = (
                tracking_path.name
            )
            snap_tracking_batches.append(batch_snap)

assert snap_tracking_batches, (
    "No snap tracking records were extracted."
)

snap_tracking = (
    pd.concat(
        snap_tracking_batches,
        ignore_index=True,
    )
    .sort_values(
        [
            "gameId",
            "playId",
            "frameId",
            "team",
            "nflId",
        ],
        na_position="last",
    )
    .reset_index(drop=True)
)

assert len(snap_tracking) == EXPECTED_SNAP_TRACKING_ROWS

snap_extraction_summary = pd.Series(
    {
        "source_files": (
            snap_tracking["source_file"].nunique()
        ),
        "snap_tracking_rows": len(snap_tracking),
        "unique_plays": (
            snap_tracking[SNAP_FRAME_KEY]
            .drop_duplicates()
            .shape[0]
        ),
        "expected_rows": EXPECTED_SNAP_TRACKING_ROWS,
    },
    name="value",
)

print(snap_extraction_summary)

source_files               8
snap_tracking_rows    196213
unique_plays            8531
expected_rows         196213
Name: value, dtype: int64


In [16]:
# Stage 4.9 — Validate snap tracking integrity

FOOTBALL_TEAM_VALUE = "football"
MANUAL_SNAP_EVENT = "ball_snap"

EXPECTED_PLAYERS_PER_SNAP = 22

EXPECTED_PLAYER_SNAP_ROWS = (
    EXPECTED_MODELING_PLAY_COUNT
    * EXPECTED_PLAYERS_PER_SNAP
)

football_snap_rows = snap_tracking.loc[
    snap_tracking["team"].eq(FOOTBALL_TEAM_VALUE)
].copy()

player_snap_rows = snap_tracking.loc[
    ~snap_tracking["team"].eq(FOOTBALL_TEAM_VALUE)
].copy()

rows_per_snap = (
    snap_tracking
    .groupby(SNAP_FRAME_KEY)
    .size()
)

players_per_snap = (
    player_snap_rows
    .groupby(SNAP_FRAME_KEY)
    .size()
)

football_rows_per_snap = (
    football_snap_rows
    .groupby(SNAP_FRAME_KEY)
    .size()
)

assert rows_per_snap.eq(EXPECTED_ENTITIES_PER_SNAP).all()
assert players_per_snap.eq(EXPECTED_PLAYERS_PER_SNAP).all()
assert football_rows_per_snap.eq(1).all()

assert len(player_snap_rows) == EXPECTED_PLAYER_SNAP_ROWS
assert len(football_snap_rows) == EXPECTED_MODELING_PLAY_COUNT

assert player_snap_rows["nflId"].notna().all()
assert football_snap_rows["nflId"].isna().all()

assert not player_snap_rows.duplicated(
    subset=PFF_PLAYER_PLAY_KEY
).any()

assert snap_tracking["frameId"].eq(
    snap_tracking["snap_frame_id"]
).all()

assert football_snap_rows["event"].eq(
    MANUAL_SNAP_EVENT
).all()

snap_entity_summary = pd.DataFrame(
    [
        {
            "entity_type": "players",
            "rows": len(player_snap_rows),
            "unique_plays": (
                player_snap_rows[SNAP_FRAME_KEY]
                .drop_duplicates()
                .shape[0]
            ),
            "rows_per_play": EXPECTED_PLAYERS_PER_SNAP,
        },
        {
            "entity_type": "football",
            "rows": len(football_snap_rows),
            "unique_plays": (
                football_snap_rows[SNAP_FRAME_KEY]
                .drop_duplicates()
                .shape[0]
            ),
            "rows_per_play": 1,
        },
    ]
)

print(snap_entity_summary)

  entity_type    rows  unique_plays  rows_per_play
0     players  187682          8531             22
1    football    8531          8531              1


In [17]:
# Stage 4.10 — Attach PFF roles to snap player tracking

player_snap_rows["nflId"] = (
    player_snap_rows["nflId"]
    .astype("int64")
)

snap_players_with_roles = player_snap_rows.merge(
    modeling_pff_roles[
        [
            *PFF_PLAYER_PLAY_KEY,
            "pff_role",
        ]
    ],
    on=PFF_PLAYER_PLAY_KEY,
    how="left",
    validate="one_to_one",
    indicator=True,
)

assert len(snap_players_with_roles) == (
    EXPECTED_MODELING_PLAYER_PLAY_ROWS
)

assert snap_players_with_roles["_merge"].eq("both").all()

assert snap_players_with_roles["pff_role"].notna().all()

snap_players_with_roles = (
    snap_players_with_roles
    .drop(columns="_merge")
)

snap_tracking_role_summary = (
    snap_players_with_roles["pff_role"]
    .value_counts()
    .rename_axis("pff_role")
    .reset_index(name="snap_rows")
)

expected_role_counts = (
    modeling_pff_role_summary
    .rename(columns={"rows": "expected_rows"})
)

snap_tracking_role_summary = (
    snap_tracking_role_summary
    .merge(
        expected_role_counts,
        on="pff_role",
        how="outer",
        validate="one_to_one",
    )
)

snap_tracking_role_summary["matches_expected"] = (
    snap_tracking_role_summary["snap_rows"].eq(
        snap_tracking_role_summary["expected_rows"]
    )
)

assert snap_tracking_role_summary[
    "matches_expected"
].all()

print(snap_tracking_role_summary)

     pff_role  snap_rows  expected_rows  matches_expected
0    Coverage      57582          57582              True
1        Pass       8531           8531              True
2  Pass Block      45921          45921              True
3  Pass Route      39389          39389              True
4   Pass Rush      36259          36259              True


In [18]:
# Stage 4.11 — Build snap entity tables and attach the target

EXPECTED_PRESSURE_POSITIVES = 4_214

quarterback_snap = (
    snap_players_with_roles.loc[
        snap_players_with_roles["pff_role"].eq("Pass")
    ]
    .copy()
)

pass_blocker_snap = (
    snap_players_with_roles.loc[
        snap_players_with_roles["pff_role"].eq(
            "Pass Block"
        )
    ]
    .copy()
)

pass_rusher_snap = (
    snap_players_with_roles.loc[
        snap_players_with_roles["pff_role"].eq(
            "Pass Rush"
        )
    ]
    .merge(
        pressure_labels[
            [
                *PRESSURE_LABEL_KEY,
                "pff_positionLinedUp",
                "pressure",
            ]
        ],
        on=PRESSURE_LABEL_KEY,
        how="left",
        validate="one_to_one",
        indicator="target_merge",
    )
)

assert len(quarterback_snap) == EXPECTED_MODELING_PLAY_COUNT
assert len(pass_blocker_snap) == 45_921
assert len(pass_rusher_snap) == EXPECTED_PRESSURE_LABEL_ROWS

assert pass_rusher_snap["target_merge"].eq("both").all()
assert pass_rusher_snap["pressure"].notna().all()

assert set(pass_rusher_snap["pressure"].unique()) == {0, 1}

assert int(pass_rusher_snap["pressure"].sum()) == (
    EXPECTED_PRESSURE_POSITIVES
)

pass_rusher_snap = pass_rusher_snap.drop(
    columns="target_merge"
)

snap_modeling_entity_summary = pd.DataFrame(
    [
        {
            "entity_type": "football",
            "rows": len(football_snap_rows),
            "plays": (
                football_snap_rows[SNAP_FRAME_KEY]
                .drop_duplicates()
                .shape[0]
            ),
        },
        {
            "entity_type": "quarterbacks",
            "rows": len(quarterback_snap),
            "plays": (
                quarterback_snap[SNAP_FRAME_KEY]
                .drop_duplicates()
                .shape[0]
            ),
        },
        {
            "entity_type": "pass_blockers",
            "rows": len(pass_blocker_snap),
            "plays": (
                pass_blocker_snap[SNAP_FRAME_KEY]
                .drop_duplicates()
                .shape[0]
            ),
        },
        {
            "entity_type": "pass_rushers",
            "rows": len(pass_rusher_snap),
            "plays": (
                pass_rusher_snap[SNAP_FRAME_KEY]
                .drop_duplicates()
                .shape[0]
            ),
        },
    ]
)

print(snap_modeling_entity_summary)

     entity_type   rows  plays
0       football   8531   8531
1   quarterbacks   8531   8531
2  pass_blockers  45921   8531
3   pass_rushers  36259   8531


In [22]:
# Stage 4.12 — Validate snap numeric quality and play direction

PLAYER_SNAP_NUMERIC_COLUMNS = [
    "x",
    "y",
    "s",
    "a",
    "dis",
    "o",
    "dir",
]

FOOTBALL_POSITION_COLUMNS = [
    "x",
    "y",
    "s",
    "a",
    "dis",
]

play_direction_counts = (
    snap_tracking
    .groupby(SNAP_FRAME_KEY)["playDirection"]
    .nunique()
)

assert play_direction_counts.eq(1).all()

play_direction_by_play = (
    snap_tracking[
        [
            *SNAP_FRAME_KEY,
            "playDirection",
        ]
    ]
    .drop_duplicates()
)

assert len(play_direction_by_play) == (
    EXPECTED_MODELING_PLAY_COUNT
)

assert not play_direction_by_play[
    "playDirection"
].isna().any()

assert set(
    play_direction_by_play["playDirection"].unique()
) == {"left", "right"}

assert player_snap_rows[
    PLAYER_SNAP_NUMERIC_COLUMNS
].notna().all().all()

assert football_snap_rows[
    FOOTBALL_POSITION_COLUMNS
].notna().all().all()

assert football_snap_rows["o"].isna().all()
assert football_snap_rows["dir"].isna().all()

play_direction_summary = (
    play_direction_by_play["playDirection"]
    .value_counts()
    .rename_axis("playDirection")
    .reset_index(name="plays")
)

numeric_quality_rows = []

for entity_type, entity_data in {
    "player": player_snap_rows,
    "football": football_snap_rows,
}.items():
    for column in PLAYER_SNAP_NUMERIC_COLUMNS:
        minimum = entity_data[column].min()
        maximum = entity_data[column].max()

        numeric_quality_rows.append(
            {
                "entity_type": entity_type,
                "variable": column,
                "missing": int(
                    entity_data[column].isna().sum()
                ),
                "minimum": (
                    None
                    if pd.isna(minimum)
                    else round(float(minimum), 3)
                ),
                "maximum": (
                    None
                    if pd.isna(maximum)
                    else round(float(maximum), 3)
                ),
            }
        )

snap_numeric_quality = pd.DataFrame(
    numeric_quality_rows
)

print(play_direction_summary)
print(snap_numeric_quality)

  playDirection  plays
0          left   4415
1         right   4116
   entity_type variable  missing  minimum  maximum
0       player        x        0     3.41   116.73
1       player        y        0     3.11    50.72
2       player        s        0     0.00     8.35
3       player        a        0     0.00     8.84
4       player      dis        0     0.00     1.06
5       player        o        0     0.01   360.00
6       player      dir        0     0.00   359.99
7     football        x        0     9.20   113.87
8     football        y        0    -2.52    53.90
9     football        s        0     0.00    10.83
10    football        a        0     0.00    24.59
11    football      dis        0     0.00     7.90
12    football        o     8531      NaN      NaN
13    football      dir     8531      NaN      NaN


In [23]:
# Stage 4.13 — Inspect snap boundary violations

FIELD_X_MIN = 0.0
FIELD_X_MAX = 120.0
FIELD_Y_MIN = 0.0
FIELD_Y_MAX = 160.0 / 3.0

boundary_summary_rows = []

for entity_type, entity_data in {
    "player": player_snap_rows,
    "football": football_snap_rows,
}.items():
    outside_x = ~entity_data["x"].between(
        FIELD_X_MIN,
        FIELD_X_MAX,
        inclusive="both",
    )

    outside_y = ~entity_data["y"].between(
        FIELD_Y_MIN,
        FIELD_Y_MAX,
        inclusive="both",
    )

    boundary_summary_rows.append(
        {
            "entity_type": entity_type,
            "outside_x_rows": int(outside_x.sum()),
            "outside_y_rows": int(outside_y.sum()),
            "any_boundary_rows": int(
                (outside_x | outside_y).sum()
            ),
        }
    )

snap_boundary_summary = pd.DataFrame(
    boundary_summary_rows
)

football_boundary_violations = (
    football_snap_rows.loc[
        ~football_snap_rows["x"].between(
            FIELD_X_MIN,
            FIELD_X_MAX,
            inclusive="both",
        )
        | ~football_snap_rows["y"].between(
            FIELD_Y_MIN,
            FIELD_Y_MAX,
            inclusive="both",
        ),
        [
            "actual_week",
            "gameId",
            "playId",
            "snap_frame_id",
            "x",
            "y",
            "s",
            "a",
            "dis",
            "source_file",
        ],
    ]
    .copy()
)

football_boundary_violations[
    "yards_outside_y"
] = (
    (FIELD_Y_MIN - football_boundary_violations["y"])
    .clip(lower=0)
    + (
        football_boundary_violations["y"]
        - FIELD_Y_MAX
    )
    .clip(lower=0)
).round(3)

football_boundary_violations = (
    football_boundary_violations
    .sort_values(
        "yards_outside_y",
        ascending=False,
    )
    .reset_index(drop=True)
)

assert snap_boundary_summary.loc[
    snap_boundary_summary["entity_type"].eq("player"),
    "any_boundary_rows",
].iloc[0] == 0

display(snap_boundary_summary)
football_boundary_violations.head(20)

,entity_type,outside_x_rows,outside_y_rows,any_boundary_rows
0,player,0,0,0
1,football,0,3,3


,actual_week,gameId,playId,snap_frame_id,x,y,s,a,dis,source_file,yards_outside_y
0,1,2021091212,912,7,63.57,-2.52,0.06,0.18,0.01,week1.parquet,2.520
1,2,2021091905,876,6,75.94,-2.50,0.16,0.25,0.01,week2.parquet,2.500
2,1,2021091203,1041,6,75.23,53.90,0.14,0.53,0.02,week1.parquet,0.567


In [25]:
# Stage 4.14 — Diagnose football boundary anomalies

boundary_pressure_by_play = (
    pressure_labels
    .groupby(SNAP_FRAME_KEY, as_index=False)
    .agg(
        pass_rushers=("nflId", "size"),
        pressure_positives=("pressure", "sum"),
    )
)

boundary_quarterbacks = (
    quarterback_snap[
        [
            *SNAP_FRAME_KEY,
            "nflId",
            "x",
            "y",
        ]
    ]
    .rename(
        columns={
            "nflId": "quarterback_nflId",
            "x": "quarterback_x",
            "y": "quarterback_y",
        }
    )
)

boundary_play_diagnostics = (
    football_boundary_violations[
        [
            "actual_week",
            *SNAP_FRAME_KEY,
            "snap_frame_id",
            "x",
            "y",
            "yards_outside_y",
        ]
    ]
    .rename(
        columns={
            "x": "football_x",
            "y": "football_y",
        }
    )
    .merge(
        boundary_quarterbacks,
        on=SNAP_FRAME_KEY,
        how="left",
        validate="one_to_one",
    )
    .merge(
        boundary_pressure_by_play,
        on=SNAP_FRAME_KEY,
        how="left",
        validate="one_to_one",
    )
)

boundary_play_diagnostics[
    "football_qb_x_gap"
] = (
    boundary_play_diagnostics["football_x"]
    - boundary_play_diagnostics["quarterback_x"]
).abs().round(3)

boundary_play_diagnostics[
    "football_qb_y_gap"
] = (
    boundary_play_diagnostics["football_y"]
    - boundary_play_diagnostics["quarterback_y"]
).abs().round(3)

assert len(boundary_play_diagnostics) == 3

assert boundary_play_diagnostics[
    [
        "quarterback_nflId",
        "quarterback_x",
        "quarterback_y",
        "pass_rushers",
        "pressure_positives",
    ]
].notna().all().all()

print(boundary_play_diagnostics)

   actual_week      gameId  playId  snap_frame_id  football_x  football_y  \
0            1  2021091212     912              7       63.57       -2.52   
1            2  2021091905     876              6       75.94       -2.50   
2            1  2021091203    1041              6       75.23       53.90   

   yards_outside_y  quarterback_nflId  quarterback_x  quarterback_y  \
0            2.520              47789          69.55          29.87   
1            2.500              43380          74.03          27.06   
2            0.567              38632          72.52          23.81   

   pass_rushers  pressure_positives  football_qb_x_gap  football_qb_y_gap  
0             5                   1               5.98              32.39  
1             5                   1               1.91              29.56  
2             4                   0               2.71              30.09  


### Football tracking boundary decision

Three of the 8,531 modeling plays contain a football `y` coordinate outside
the official field width at the manual snap. No player coordinates are outside
the field.

The affected football records are laterally separated from their quarterbacks
by approximately 30 yards, confirming that `football_y` is unreliable in these
three plays. Their `football_x` values remain longitudinally plausible.

The three plays contain 14 pass rushers and two positive pressure outcomes.
They are retained because the anomaly is isolated to the football's lateral
coordinate and does not invalidate the player tracking or target.

No clipping or synthetic coordinate replacement is applied. `football_y`
remains available only for auditing and will not be used to create model
features. `football_x` may be used as a longitudinal line-of-scrimmage
reference, while lateral spatial relationships will use the quarterback's
position.

### Spatial coordinate normalization

The official NFL tracking schema defines:

- `x`: position along the long axis of the field, approximately from 0 to 120 yards.
- `y`: position along the short axis, approximately from 0 to `160 / 3` yards.
- `o`: player body orientation in degrees.
- `dir`: player movement direction in degrees.
- `playDirection`: original offensive direction (`left` or `right`).

Sources:

- [NFL Big Data Bowl tracking schema](https://github.com/nfl-football-ops/Big-Data-Bowl/blob/master/schema.md)
- [Official NFL Big Data Bowl repository](https://github.com/nfl-football-ops/Big-Data-Bowl)

To make every offense advance toward increasing `x`, plays whose original
`playDirection` is `left` will be rotated 180 degrees around the center of
the field:

- `x_norm = 120 - x`
- `y_norm = (160 / 3) - y`
- `o_norm = (o + 180) mod 360`
- `dir_norm = (dir + 180) mod 360`

Right-moving plays retain their original coordinates and angles, except
that angles will be expressed consistently in the interval `[0, 360)`.

A 180-degree rotation preserves distances, spatial relationships, and
offensive left/right handedness. Variables `s`, `a`, and `dis` remain
unchanged because their magnitudes do not depend on field orientation.

Raw coordinates will be retained for traceability, while normalized
coordinates will be used to construct spatial predictors.

The three football `y` boundary anomalies previously identified remain in
the dataset. `football_y` and its normalized version are audit-only fields
and will not be used to generate model predictors. `football_x` may be used
as a longitudinal snap reference.

In [26]:
FIELD_LENGTH_YARDS = 120.0
FIELD_WIDTH_YARDS = 160.0 / 3.0
HALF_TURN_DEGREES = 180.0
FULL_TURN_DEGREES = 360.0


def normalize_snap_coordinates(dataframe):
    """Rotate left-moving plays into a common right-moving reference."""
    required_columns = {
        "x",
        "y",
        "o",
        "dir",
        "playDirection",
    }
    missing_columns = required_columns.difference(dataframe.columns)

    assert not missing_columns, (
        f"Missing required columns: {sorted(missing_columns)}"
    )
    assert dataframe["playDirection"].notna().all()
    assert set(dataframe["playDirection"].unique()).issubset(
        {"left", "right"}
    )

    normalized = dataframe.copy()
    left_mask = normalized["playDirection"].eq("left")

    normalized["x_norm"] = normalized["x"].where(
        ~left_mask,
        FIELD_LENGTH_YARDS - normalized["x"],
    )
    normalized["y_norm"] = normalized["y"].where(
        ~left_mask,
        FIELD_WIDTH_YARDS - normalized["y"],
    )

    rotation_degrees = left_mask.astype("int16") * HALF_TURN_DEGREES

    normalized["o_norm"] = (
        normalized["o"] + rotation_degrees
    ).mod(FULL_TURN_DEGREES)

    normalized["dir_norm"] = (
        normalized["dir"] + rotation_degrees
    ).mod(FULL_TURN_DEGREES)

    assert len(normalized) == len(dataframe)

    return normalized


football_snap_normalized = normalize_snap_coordinates(
    football_snap_rows
)
quarterback_snap_normalized = normalize_snap_coordinates(
    quarterback_snap
)
pass_blocker_snap_normalized = normalize_snap_coordinates(
    pass_blocker_snap
)
pass_rusher_snap_normalized = normalize_snap_coordinates(
    pass_rusher_snap
)

normalized_entity_tables = {
    "football": football_snap_normalized,
    "quarterbacks": quarterback_snap_normalized,
    "pass_blockers": pass_blocker_snap_normalized,
    "pass_rushers": pass_rusher_snap_normalized,
}

normalization_summary = pd.DataFrame(
    [
        {
            "entity_type": entity_type,
            "rows": len(dataframe),
            "left_rows": dataframe["playDirection"].eq("left").sum(),
            "right_rows": dataframe["playDirection"].eq("right").sum(),
            "x_norm_min": dataframe["x_norm"].min(),
            "x_norm_max": dataframe["x_norm"].max(),
            "y_norm_min": dataframe["y_norm"].min(),
            "y_norm_max": dataframe["y_norm"].max(),
        }
        for entity_type, dataframe in normalized_entity_tables.items()
    ]
).round(3)

normalization_summary

,entity_type,rows,left_rows,right_rows,x_norm_min,x_norm_max,y_norm_min,y_norm_max
0,football,8531,4415,4116,6.13,109.66,-2.500,55.853
1,quarterbacks,8531,4415,4116,5.54,108.11,22.203,31.080
2,pass_blockers,45921,23720,22201,4.08,109.36,15.150,43.793
3,pass_rushers,36259,18780,17479,11.02,113.02,8.810,46.830


In [27]:
import numpy as np


PLAY_KEYS = ["actual_week", "gameId", "playId"]

raw_entity_tables = {
    "football": football_snap_rows,
    "quarterbacks": quarterback_snap,
    "pass_blockers": pass_blocker_snap,
    "pass_rushers": pass_rusher_snap,
}

# Confirm that every original column remains unchanged.
for entity_type, raw_dataframe in raw_entity_tables.items():
    normalized_dataframe = normalized_entity_tables[entity_type]

    pd.testing.assert_frame_equal(
        raw_dataframe.reset_index(drop=True),
        normalized_dataframe[
            raw_dataframe.columns
        ].reset_index(drop=True),
    )

    left_mask = raw_dataframe["playDirection"].eq("left")
    rotation_degrees = (
        left_mask.astype("int16") * HALF_TURN_DEGREES
    )

    expected_x = raw_dataframe["x"].where(
        ~left_mask,
        FIELD_LENGTH_YARDS - raw_dataframe["x"],
    )
    expected_y = raw_dataframe["y"].where(
        ~left_mask,
        FIELD_WIDTH_YARDS - raw_dataframe["y"],
    )
    expected_o = (
        raw_dataframe["o"] + rotation_degrees
    ).mod(FULL_TURN_DEGREES)
    expected_dir = (
        raw_dataframe["dir"] + rotation_degrees
    ).mod(FULL_TURN_DEGREES)

    np.testing.assert_allclose(
        normalized_dataframe["x_norm"],
        expected_x,
        atol=1e-12,
    )
    np.testing.assert_allclose(
        normalized_dataframe["y_norm"],
        expected_y,
        atol=1e-12,
    )
    np.testing.assert_allclose(
        normalized_dataframe["o_norm"],
        expected_o,
        atol=1e-12,
        equal_nan=True,
    )
    np.testing.assert_allclose(
        normalized_dataframe["dir_norm"],
        expected_dir,
        atol=1e-12,
        equal_nan=True,
    )


player_tables = {
    "quarterbacks": quarterback_snap_normalized,
    "pass_blockers": pass_blocker_snap_normalized,
    "pass_rushers": pass_rusher_snap_normalized,
}

player_boundary_violations = sum(
    (
        ~dataframe["x_norm"].between(
            0.0,
            FIELD_LENGTH_YARDS,
        )
        | ~dataframe["y_norm"].between(
            0.0,
            FIELD_WIDTH_YARDS,
        )
    ).sum()
    for dataframe in player_tables.values()
)

player_angle_violations = sum(
    (
        ~dataframe["o_norm"].between(
            0.0,
            FULL_TURN_DEGREES,
            inclusive="left",
        )
        | ~dataframe["dir_norm"].between(
            0.0,
            FULL_TURN_DEGREES,
            inclusive="left",
        )
    ).sum()
    for dataframe in player_tables.values()
)

football_y_violations = (
    ~football_snap_normalized["y_norm"].between(
        0.0,
        FIELD_WIDTH_YARDS,
    )
).sum()

assert player_boundary_violations == 0
assert player_angle_violations == 0
assert football_y_violations == 3

# Confirm that pressure labels remain unchanged.
pd.testing.assert_series_equal(
    pass_rusher_snap["pressure"].reset_index(drop=True),
    pass_rusher_snap_normalized[
        "pressure"
    ].reset_index(drop=True),
)

assert int(pass_rusher_snap_normalized["pressure"].sum()) == 4214

# Confirm that rotation preserves distances to the quarterback.
quarterback_reference = quarterback_snap_normalized[
    PLAY_KEYS + ["x", "y", "x_norm", "y_norm"]
].rename(
    columns={
        "x": "quarterback_x",
        "y": "quarterback_y",
        "x_norm": "quarterback_x_norm",
        "y_norm": "quarterback_y_norm",
    }
)

assert not quarterback_reference.duplicated(PLAY_KEYS).any()

distance_sources = {
    "football_to_quarterback": football_snap_normalized,
    "pass_blocker_to_quarterback": pass_blocker_snap_normalized,
    "pass_rusher_to_quarterback": pass_rusher_snap_normalized,
}

distance_results = []

for relationship, dataframe in distance_sources.items():
    comparison = dataframe[
        PLAY_KEYS + ["x", "y", "x_norm", "y_norm"]
    ].merge(
        quarterback_reference,
        on=PLAY_KEYS,
        how="left",
        validate="many_to_one",
    )

    raw_distance = np.hypot(
        comparison["x"] - comparison["quarterback_x"],
        comparison["y"] - comparison["quarterback_y"],
    )
    normalized_distance = np.hypot(
        comparison["x_norm"]
        - comparison["quarterback_x_norm"],
        comparison["y_norm"]
        - comparison["quarterback_y_norm"],
    )

    maximum_error = np.abs(
        raw_distance - normalized_distance
    ).max()

    assert np.allclose(
        raw_distance,
        normalized_distance,
        atol=1e-10,
        rtol=0.0,
    )

    distance_results.append(
        {
            "relationship": relationship,
            "rows_checked": len(comparison),
            "maximum_absolute_error": maximum_error,
        }
    )

distance_validation_summary = pd.DataFrame(
    distance_results
)

integrity_summary = pd.DataFrame(
    [
        {
            "check": "Original data preserved",
            "status": "PASS",
            "result": "All original columns and rows are unchanged",
        },
        {
            "check": "Player spatial boundaries",
            "status": "PASS",
            "result": f"{player_boundary_violations} violations",
        },
        {
            "check": "Player angular boundaries",
            "status": "PASS",
            "result": f"{player_angle_violations} violations",
        },
        {
            "check": "Football audit anomalies",
            "status": "PASS",
            "result": f"{football_y_violations} retained anomalies",
        },
        {
            "check": "Pressure target preserved",
            "status": "PASS",
            "result": "36,259 rows; 4,214 positives",
        },
        {
            "check": "Distances preserved",
            "status": "PASS",
            "result": (
                f"{sum(item['rows_checked'] for item in distance_results):,} "
                "relationships checked"
            ),
        },
    ]
)

display(integrity_summary)
display(distance_validation_summary)

,check,status,result
0,Original data preserved,PASS,All original columns and rows are unchanged
1,Player spatial boundaries,PASS,0 violations
2,Player angular boundaries,PASS,0 violations
3,Football audit anomalies,PASS,3 retained anomalies
4,Pressure target preserved,PASS,"36,259 rows; 4,214 positives"
5,Distances preserved,PASS,"90,711 relationships checked"


,relationship,rows_checked,maximum_absolute_error
0,football_to_quarterback,8531,1.509903e-14
1,pass_blocker_to_quarterback,45921,1.509903e-14
2,pass_rusher_to_quarterback,36259,1.598721e-14


In [29]:
PLAY_KEYS = ["actual_week", "gameId", "playId"]
RUSHER_KEYS = PLAY_KEYS + ["pass_rusher_nfl_id"]

pass_rusher_base = pass_rusher_snap_normalized[
    [
        "actual_week",
        "gameId",
        "playId",
        "nflId",
        "pff_positionLinedUp",
        "pressure",
        "playDirection",
        "x_norm",
        "y_norm",
        "s",
        "a",
        "dis",
        "o_norm",
        "dir_norm",
    ]
].rename(
    columns={
        "nflId": "pass_rusher_nfl_id",
        "pff_positionLinedUp": "rusher_position_lined_up",
        "playDirection": "original_play_direction",
        "x_norm": "rusher_x",
        "y_norm": "rusher_y",
        "s": "rusher_speed",
        "a": "rusher_acceleration",
        "dis": "rusher_displacement",
        "o_norm": "rusher_orientation",
        "dir_norm": "rusher_direction",
    }
)

quarterback_reference = quarterback_snap_normalized[
    PLAY_KEYS
    + [
        "nflId",
        "x_norm",
        "y_norm",
        "s",
        "a",
        "dis",
        "o_norm",
        "dir_norm",
    ]
].rename(
    columns={
        "nflId": "quarterback_nfl_id",
        "x_norm": "quarterback_x",
        "y_norm": "quarterback_y",
        "s": "quarterback_speed",
        "a": "quarterback_acceleration",
        "dis": "quarterback_displacement",
        "o_norm": "quarterback_orientation",
        "dir_norm": "quarterback_direction",
    }
)

football_reference = football_snap_normalized[
    PLAY_KEYS + ["x_norm"]
].rename(
    columns={
        "x_norm": "football_x",
    }
)

assert not pass_rusher_base.duplicated(RUSHER_KEYS).any()
assert not quarterback_reference.duplicated(PLAY_KEYS).any()
assert not football_reference.duplicated(PLAY_KEYS).any()

analytical_base = (
    pass_rusher_base.merge(
        quarterback_reference,
        on=PLAY_KEYS,
        how="left",
        validate="many_to_one",
    )
    .merge(
        football_reference,
        on=PLAY_KEYS,
        how="left",
        validate="many_to_one",
    )
)

excluded_source_columns = {
    "pff_hit",
    "pff_hurry",
    "pff_sack",
    "pff_beatenByDefender",
    "pff_hitAllowed",
    "pff_hurryAllowed",
    "pff_sackAllowed",
    "pff_nflIdBlockedPlayer",
    "pff_blockType",
    "pff_backFieldBlock",
}

excluded_columns_present = excluded_source_columns.intersection(
    analytical_base.columns
)

assert len(analytical_base) == 36259
assert not analytical_base.duplicated(RUSHER_KEYS).any()
assert analytical_base.drop_duplicates(PLAY_KEYS).shape[0] == 8531
assert analytical_base.isna().sum().sum() == 0
assert set(analytical_base["pressure"].unique()).issubset({0, 1})
assert int(analytical_base["pressure"].sum()) == 4214
assert not excluded_columns_present

analytical_base_summary = pd.DataFrame(
    [
        {
            "metric": "analytical_rows",
            "value": len(analytical_base),
        },
        {
            "metric": "columns",
            "value": analytical_base.shape[1],
        },
        {
            "metric": "unique_rusher_keys",
            "value": analytical_base[RUSHER_KEYS]
            .drop_duplicates()
            .shape[0],
        },
        {
            "metric": "modeling_plays",
            "value": analytical_base[PLAY_KEYS]
            .drop_duplicates()
            .shape[0],
        },
        {
            "metric": "pressure_positives",
            "value": int(analytical_base["pressure"].sum()),
        },
        {
            "metric": "pressure_prevalence_pct",
            "value": round(
                analytical_base["pressure"].mean() * 100,
                3,
            ),
        },
        {
            "metric": "missing_values",
            "value": int(analytical_base.isna().sum().sum()),
        },
        {
            "metric": "excluded_source_columns_present",
            "value": len(excluded_columns_present),
        },
    ]
)

print(analytical_base_summary)

                            metric      value
0                  analytical_rows  36259.000
1                          columns     23.000
2               unique_rusher_keys  36259.000
3                   modeling_plays   8531.000
4               pressure_positives   4214.000
5          pressure_prevalence_pct     11.622
6                   missing_values      0.000
7  excluded_source_columns_present      0.000


In [30]:
analytical_table = analytical_base.copy()

# Signed longitudinal and lateral relationships.
analytical_table["rusher_qb_longitudinal_gap"] = (
    analytical_table["rusher_x"]
    - analytical_table["quarterback_x"]
)

analytical_table["rusher_qb_lateral_offset"] = (
    analytical_table["rusher_y"]
    - analytical_table["quarterback_y"]
)

# Absolute lateral separation.
analytical_table["rusher_qb_absolute_lateral_gap"] = (
    analytical_table["rusher_qb_lateral_offset"].abs()
)

# Straight-line distance between rusher and quarterback.
analytical_table["rusher_qb_distance"] = np.hypot(
    analytical_table["rusher_qb_longitudinal_gap"],
    analytical_table["rusher_qb_lateral_offset"],
)

# Signed longitudinal position relative to the ball.
analytical_table["rusher_x_relative_to_ball"] = (
    analytical_table["rusher_x"]
    - analytical_table["football_x"]
)

# How far the quarterback is positioned behind the ball.
analytical_table["quarterback_depth_behind_ball"] = (
    analytical_table["football_x"]
    - analytical_table["quarterback_x"]
)

SPATIAL_FEATURE_COLUMNS = [
    "rusher_qb_longitudinal_gap",
    "rusher_qb_lateral_offset",
    "rusher_qb_absolute_lateral_gap",
    "rusher_qb_distance",
    "rusher_x_relative_to_ball",
    "quarterback_depth_behind_ball",
]

assert len(analytical_table) == 36259
assert not analytical_table.duplicated(RUSHER_KEYS).any()
assert analytical_table[SPATIAL_FEATURE_COLUMNS].notna().all().all()
assert np.isfinite(
    analytical_table[SPATIAL_FEATURE_COLUMNS].to_numpy()
).all()
assert analytical_table["rusher_qb_distance"].ge(0.0).all()

np.testing.assert_allclose(
    analytical_table["rusher_qb_distance"],
    np.hypot(
        analytical_table["rusher_qb_longitudinal_gap"],
        analytical_table["rusher_qb_lateral_offset"],
    ),
    atol=1e-12,
)

pd.testing.assert_series_equal(
    analytical_table["pressure"],
    analytical_base["pressure"],
)

assert "football_y" not in analytical_table.columns
assert "football_y_norm" not in analytical_table.columns

spatial_feature_quality = pd.DataFrame(
    [
        {
            "feature": feature,
            "unit": "yards",
            "missing": analytical_table[feature].isna().sum(),
            "non_finite": (
                ~np.isfinite(analytical_table[feature])
            ).sum(),
            "minimum": analytical_table[feature].min(),
            "median": analytical_table[feature].median(),
            "maximum": analytical_table[feature].max(),
        }
        for feature in SPATIAL_FEATURE_COLUMNS
    ]
).round(3)

print(spatial_feature_quality)

                          feature   unit  missing  non_finite  minimum  \
0      rusher_qb_longitudinal_gap  yards        0           0   -3.340   
1        rusher_qb_lateral_offset  yards        0           0  -15.010   
2  rusher_qb_absolute_lateral_gap  yards        0           0    0.000   
3              rusher_qb_distance  yards        0           0    1.569   
4       rusher_x_relative_to_ball  yards        0           0   -4.830   
5   quarterback_depth_behind_ball  yards        0           0   -2.710   

   median  maximum  
0   5.490   15.370  
1  -0.100   16.950  
2   3.190   16.950  
3   6.418   17.544  
4   1.000   11.300  
5   4.390    6.600  


In [32]:
play_level_spatial_data = (
    analytical_table[
        PLAY_KEYS
        + [
            "quarterback_depth_behind_ball",
        ]
    ]
    .drop_duplicates(PLAY_KEYS)
    .reset_index(drop=True)
)

assert len(play_level_spatial_data) == 8531
assert not play_level_spatial_data.duplicated(PLAY_KEYS).any()

diagnostic_inputs = [
    {
        "feature": "rusher_qb_longitudinal_gap",
        "observation_unit": "pass_rusher",
        "dataframe": analytical_table,
    },
    {
        "feature": "rusher_x_relative_to_ball",
        "observation_unit": "pass_rusher",
        "dataframe": analytical_table,
    },
    {
        "feature": "quarterback_depth_behind_ball",
        "observation_unit": "play",
        "dataframe": play_level_spatial_data,
    },
]

spatial_sign_results = []

for diagnostic in diagnostic_inputs:
    feature = diagnostic["feature"]
    observation_unit = diagnostic["observation_unit"]
    dataframe = diagnostic["dataframe"]

    negative_mask = dataframe[feature].lt(0.0)
    affected_plays = (
        dataframe.loc[negative_mask, PLAY_KEYS]
        .drop_duplicates()
        .shape[0]
    )
    total_plays = dataframe[PLAY_KEYS].drop_duplicates().shape[0]

    spatial_sign_results.append(
        {
            "feature": feature,
            "observation_unit": observation_unit,
            "observations": len(dataframe),
            "negative_values": int(negative_mask.sum()),
            "negative_pct": round(
                negative_mask.mean() * 100,
                3,
            ),
            "affected_plays": affected_plays,
            "affected_plays_pct": round(
                affected_plays / total_plays * 100,
                3,
            ),
            "minimum": dataframe[feature].min(),
            "median": dataframe[feature].median(),
        }
    )

spatial_sign_diagnostics = pd.DataFrame(
    spatial_sign_results
).round(3)

assert spatial_sign_diagnostics["observations"].gt(0).all()

print(spatial_sign_diagnostics)

                         feature observation_unit  observations  \
0     rusher_qb_longitudinal_gap      pass_rusher         36259   
1      rusher_x_relative_to_ball      pass_rusher         36259   
2  quarterback_depth_behind_ball             play          8531   

   negative_values  negative_pct  affected_plays  affected_plays_pct  minimum  \
0                2         0.006               2               0.023    -3.34   
1               78         0.215              54               0.633    -4.83   
2                3         0.035               3               0.035    -2.71   

   median  
0    5.49  
1    1.00  
2    4.40  


In [33]:
negative_rusher_qb_mask = (
    analytical_table["rusher_qb_longitudinal_gap"] < 0.0
)
negative_rusher_ball_mask = (
    analytical_table["rusher_x_relative_to_ball"] < 0.0
)
negative_quarterback_ball_mask = (
    analytical_table["quarterback_depth_behind_ball"] < 0.0
)

rare_rusher_qb_cases = (
    analytical_table.loc[
        negative_rusher_qb_mask,
        [
            "actual_week",
            "gameId",
            "playId",
            "pass_rusher_nfl_id",
            "rusher_position_lined_up",
            "original_play_direction",
            "rusher_x",
            "quarterback_x",
            "football_x",
            "rusher_qb_longitudinal_gap",
            "rusher_qb_lateral_offset",
            "rusher_qb_distance",
            "rusher_x_relative_to_ball",
        ],
    ]
    .sort_values("rusher_qb_longitudinal_gap")
    .reset_index(drop=True)
)

rare_quarterback_ball_cases = (
    analytical_table.loc[
        negative_quarterback_ball_mask,
        [
            "actual_week",
            "gameId",
            "playId",
            "quarterback_nfl_id",
            "original_play_direction",
            "quarterback_x",
            "football_x",
            "quarterback_depth_behind_ball",
        ],
    ]
    .drop_duplicates(PLAY_KEYS)
    .sort_values("quarterback_depth_behind_ball")
    .reset_index(drop=True)
)

negative_rusher_ball_cases = analytical_table.loc[
    negative_rusher_ball_mask
].copy()

position_row_summary = (
    negative_rusher_ball_cases.groupby(
        "rusher_position_lined_up",
        dropna=False,
    )
    .agg(
        negative_rusher_rows=(
            "pass_rusher_nfl_id",
            "size",
        ),
        minimum_relative_x=(
            "rusher_x_relative_to_ball",
            "min",
        ),
        median_relative_x=(
            "rusher_x_relative_to_ball",
            "median",
        ),
        maximum_relative_x=(
            "rusher_x_relative_to_ball",
            "max",
        ),
    )
)

position_play_summary = (
    negative_rusher_ball_cases[
        ["rusher_position_lined_up"] + PLAY_KEYS
    ]
    .drop_duplicates()
    .groupby(
        "rusher_position_lined_up",
        dropna=False,
    )
    .size()
    .rename("affected_plays")
)

negative_rusher_ball_position_summary = (
    position_row_summary.join(position_play_summary)
    .reset_index()
    .sort_values(
        ["negative_rusher_rows", "affected_plays"],
        ascending=False,
    )
    .reset_index(drop=True)
    .round(3)
)

assert len(rare_rusher_qb_cases) == 2
assert len(rare_quarterback_ball_cases) == 3
assert len(negative_rusher_ball_cases) == 78
assert (
    negative_rusher_ball_position_summary[
        "negative_rusher_rows"
    ].sum()
    == 78
)

display(rare_rusher_qb_cases)
display(rare_quarterback_ball_cases)
display(negative_rusher_ball_position_summary)

,actual_week,gameId,playId,pass_rusher_nfl_id,rusher_position_lined_up,original_play_direction,rusher_x,quarterback_x,football_x,rusher_qb_longitudinal_gap,rusher_qb_lateral_offset,rusher_qb_distance,rusher_x_relative_to_ball
0,1,2021091204,2699,45011,NT,right,39.01,42.35,43.84,-3.34,-0.04,3.340240,-4.83
1,1,2021091204,2652,45011,DLT,right,33.21,33.22,34.89,-0.01,-2.86,2.860017,-1.68


,actual_week,gameId,playId,quarterback_nfl_id,original_play_direction,quarterback_x,football_x,quarterback_depth_behind_ball
0,1,2021091203,1041,38632,left,47.48,44.77,-2.71
1,1,2021091201,500,46076,left,51.39,51.34,-0.05
2,1,2021091210,1020,53444,right,94.37,94.35,-0.02


,rusher_position_lined_up,negative_rusher_rows,minimum_relative_x,median_relative_x,maximum_relative_x,affected_plays
0,REO,20,-0.52,-0.120,-0.01,20
1,LEO,17,-0.77,-0.080,-0.01,17
2,DLT,8,-1.68,-0.270,-0.05,8
3,LE,7,-0.86,-0.240,-0.05,7
4,RE,6,-0.42,-0.135,-0.05,6
5,DRT,5,-0.42,-0.110,-0.03,5
6,ROLB,5,-1.64,-0.150,-0.03,5
7,LOLB,4,-0.77,-0.240,-0.04,4
8,NT,4,-4.83,-0.425,-0.09,4
9,NRT,1,-0.08,-0.080,-0.08,1


### Decision on signed spatial exceptions

Signed spatial features were reviewed before continuing feature engineering.

Observed exceptions:

- `rusher_qb_longitudinal_gap < 0`:
  2 of 36,259 pass-rusher observations (0.006%).
  One value is effectively zero (`-0.01` yards); the other is an unusual
  `-3.34`-yard alignment.

- `rusher_x_relative_to_ball < 0`:
  78 of 36,259 pass-rusher observations (0.215%), affecting 54 plays.
  Most position groups have small negative median values and correspond to
  plausible defensive-front alignments.

- `quarterback_depth_behind_ball < 0`:
  3 of 8,531 plays (0.035%).
  Two values are effectively zero (`-0.05` and `-0.02` yards).
  The remaining value (`-2.71` yards) belongs to game `2021091203`,
  play `1041`, which was already documented as a football-tracking anomaly.

Project decision:

1. Retain all 36,259 analytical rows.
2. Preserve signed values without clipping, absolute-value replacement,
   imputation, or row deletion.
3. Treat `football_x` as the ball's longitudinal position at the tagged snap
   frame, not as an infallible measurement of the line of scrimmage.
4. Do not create rare anomaly indicators as model predictors, because their
   extremely low frequency could encourage overfitting.
5. Do not use `pressure` or any outcome component when making spatial
   quality-control decisions.
6. Preserve identifiers only for traceability and future investigation.

These decisions maintain reproducibility and avoid introducing subjective
corrections unsupported by the source data.


In [35]:
BLOCKER_KEYS = PLAY_KEYS + ["pass_blocker_nfl_id"]
PAIR_KEYS = RUSHER_KEYS + ["pass_blocker_nfl_id"]

rusher_geometry = analytical_table[
    RUSHER_KEYS
    + [
        "rusher_x",
        "rusher_y",
    ]
].copy()

blocker_reference = pass_blocker_snap_normalized[
    PLAY_KEYS
    + [
        "nflId",
        "x_norm",
        "y_norm",
        "s",
        "a",
        "dis",
        "o_norm",
        "dir_norm",
    ]
].rename(
    columns={
        "nflId": "pass_blocker_nfl_id",
        "x_norm": "blocker_x",
        "y_norm": "blocker_y",
        "s": "blocker_speed",
        "a": "blocker_acceleration",
        "dis": "blocker_displacement",
        "o_norm": "blocker_orientation",
        "dir_norm": "blocker_direction",
    }
)

assert not rusher_geometry.duplicated(RUSHER_KEYS).any()
assert not blocker_reference.duplicated(BLOCKER_KEYS).any()

play_entity_counts = (
    rusher_geometry.groupby(PLAY_KEYS)
    .size()
    .rename("pass_rushers")
    .to_frame()
    .join(
        blocker_reference.groupby(PLAY_KEYS)
        .size()
        .rename("pass_blockers"),
        how="outer",
    )
    .reset_index()
)

assert len(play_entity_counts) == 8531
assert play_entity_counts[
    ["pass_rushers", "pass_blockers"]
].notna().all().all()

play_entity_counts["expected_candidate_pairs"] = (
    play_entity_counts["pass_rushers"]
    * play_entity_counts["pass_blockers"]
)

expected_candidate_pairs = int(
    play_entity_counts["expected_candidate_pairs"].sum()
)

rusher_blocker_candidates = rusher_geometry.merge(
    blocker_reference,
    on=PLAY_KEYS,
    how="left",
    validate="many_to_many",
)

rusher_blocker_candidates[
    "rusher_blocker_longitudinal_offset"
] = (
    rusher_blocker_candidates["blocker_x"]
    - rusher_blocker_candidates["rusher_x"]
)

rusher_blocker_candidates[
    "rusher_blocker_lateral_offset"
] = (
    rusher_blocker_candidates["blocker_y"]
    - rusher_blocker_candidates["rusher_y"]
)

rusher_blocker_candidates["rusher_blocker_distance"] = np.hypot(
    rusher_blocker_candidates[
        "rusher_blocker_longitudinal_offset"
    ],
    rusher_blocker_candidates[
        "rusher_blocker_lateral_offset"
    ],
)

duplicated_pair_keys = int(
    rusher_blocker_candidates.duplicated(PAIR_KEYS).sum()
)
candidate_missing_values = int(
    rusher_blocker_candidates.isna().sum().sum()
)

assert len(rusher_blocker_candidates) == expected_candidate_pairs
assert duplicated_pair_keys == 0
assert candidate_missing_values == 0
assert "pressure" not in rusher_blocker_candidates.columns
assert np.isfinite(
    rusher_blocker_candidates["rusher_blocker_distance"]
).all()
assert rusher_blocker_candidates[
    "rusher_blocker_distance"
].ge(0.0).all()

pair_construction_summary = pd.DataFrame(
    [
        {
            "metric": "modeling_plays",
            "value": len(play_entity_counts),
        },
        {
            "metric": "unique_pass_rushers",
            "value": len(rusher_geometry),
        },
        {
            "metric": "unique_pass_blockers",
            "value": len(blocker_reference),
        },
        {
            "metric": "candidate_pairs",
            "value": len(rusher_blocker_candidates),
        },
        {
            "metric": "expected_candidate_pairs",
            "value": expected_candidate_pairs,
        },
        {
            "metric": "minimum_blockers_per_play",
            "value": play_entity_counts["pass_blockers"].min(),
        },
        {
            "metric": "median_blockers_per_play",
            "value": play_entity_counts["pass_blockers"].median(),
        },
        {
            "metric": "maximum_blockers_per_play",
            "value": play_entity_counts["pass_blockers"].max(),
        },
        {
            "metric": "duplicated_pair_keys",
            "value": duplicated_pair_keys,
        },
        {
            "metric": "missing_values",
            "value": candidate_missing_values,
        },
    ]
)

print(pair_construction_summary)

                      metric     value
0             modeling_plays    8531.0
1        unique_pass_rushers   36259.0
2       unique_pass_blockers   45921.0
3            candidate_pairs  196361.0
4   expected_candidate_pairs  196361.0
5  minimum_blockers_per_play       5.0
6   median_blockers_per_play       5.0
7  maximum_blockers_per_play       9.0
8       duplicated_pair_keys       0.0
9             missing_values       0.0


In [38]:
distance_column = "rusher_blocker_distance"

minimum_distance_by_candidate = (
    rusher_blocker_candidates.groupby(RUSHER_KEYS)[
        distance_column
    ]
    .transform("min")
)

is_minimum_distance = np.isclose(
    rusher_blocker_candidates[distance_column],
    minimum_distance_by_candidate,
    atol=1e-12,
    rtol=0.0,
)

minimum_tie_counts = (
    rusher_blocker_candidates.loc[is_minimum_distance]
    .groupby(RUSHER_KEYS)
    .size()
)

rushers_with_tied_minimum = int(
    minimum_tie_counts.gt(1).sum()
)

sorted_blocker_candidates = (
    rusher_blocker_candidates.sort_values(
        RUSHER_KEYS
        + [
            distance_column,
            "pass_blocker_nfl_id",
        ],
        ascending=True,
        kind="mergesort",
    )
)

nearest_blocker_reference = (
    sorted_blocker_candidates.drop_duplicates(
        RUSHER_KEYS,
        keep="first",
    )[
        RUSHER_KEYS
        + [
            "pass_blocker_nfl_id",
            "blocker_x",
            "blocker_y",
            "blocker_speed",
            "blocker_acceleration",
            "blocker_displacement",
            "blocker_orientation",
            "blocker_direction",
            "rusher_blocker_longitudinal_offset",
            "rusher_blocker_lateral_offset",
            "rusher_blocker_distance",
        ]
    ]
    .rename(
        columns={
            "pass_blocker_nfl_id": "nearest_blocker_nfl_id",
            "blocker_x": "nearest_blocker_x",
            "blocker_y": "nearest_blocker_y",
            "blocker_speed": "nearest_blocker_speed",
            "blocker_acceleration": (
                "nearest_blocker_acceleration"
            ),
            "blocker_displacement": (
                "nearest_blocker_displacement"
            ),
            "blocker_orientation": (
                "nearest_blocker_orientation"
            ),
            "blocker_direction": "nearest_blocker_direction",
            "rusher_blocker_longitudinal_offset": (
                "nearest_blocker_longitudinal_offset"
            ),
            "rusher_blocker_lateral_offset": (
                "nearest_blocker_lateral_offset"
            ),
            "rusher_blocker_distance": (
                "nearest_blocker_distance"
            ),
        }
    )
    .reset_index(drop=True)
)

minimum_distance_reference = (
    rusher_blocker_candidates.groupby(
        RUSHER_KEYS,
        as_index=False,
    )[distance_column]
    .min()
    .rename(
        columns={
            distance_column: "expected_minimum_distance",
        }
    )
)

nearest_distance_check = nearest_blocker_reference.merge(
    minimum_distance_reference,
    on=RUSHER_KEYS,
    how="left",
    validate="one_to_one",
)

np.testing.assert_allclose(
    nearest_distance_check["nearest_blocker_distance"],
    nearest_distance_check["expected_minimum_distance"],
    atol=1e-12,
)

duplicated_nearest_keys = int(
    nearest_blocker_reference.duplicated(RUSHER_KEYS).sum()
)
nearest_missing_values = int(
    nearest_blocker_reference.isna().sum().sum()
)
zero_distance_rows = int(
    nearest_blocker_reference[
        "nearest_blocker_distance"
    ].le(1e-12).sum()
)

assert len(nearest_blocker_reference) == 36259
assert duplicated_nearest_keys == 0
assert nearest_missing_values == 0
assert nearest_blocker_reference[
    "nearest_blocker_distance"
].ge(0.0).all()
assert "pressure" not in nearest_blocker_reference.columns

nearest_blocker_selection_summary = pd.DataFrame(
    [
        {
            "metric": "nearest_blocker_rows",
            "value": len(nearest_blocker_reference),
        },
        {
            "metric": "unique_rusher_keys",
            "value": nearest_blocker_reference[
                RUSHER_KEYS
            ].drop_duplicates().shape[0],
        },
        {
            "metric": "duplicated_rusher_keys",
            "value": duplicated_nearest_keys,
        },
        {
            "metric": "missing_values",
            "value": nearest_missing_values,
        },
        {
            "metric": "rushers_with_tied_minimum",
            "value": rushers_with_tied_minimum,
        },
        {
            "metric": "zero_distance_rows",
            "value": zero_distance_rows,
        },
        {
            "metric": "minimum_distance",
            "value": nearest_blocker_reference[
                "nearest_blocker_distance"
            ].min(),
        },
        {
            "metric": "median_distance",
            "value": nearest_blocker_reference[
                "nearest_blocker_distance"
            ].median(),
        },
        {
            "metric": "maximum_distance",
            "value": nearest_blocker_reference[
                "nearest_blocker_distance"
            ].max(),
        },
    ]
).round(3)

print(nearest_blocker_selection_summary)

                      metric      value
0       nearest_blocker_rows  36259.000
1         unique_rusher_keys  36259.000
2     duplicated_rusher_keys      0.000
3             missing_values      0.000
4  rushers_with_tied_minimum      1.000
5         zero_distance_rows      0.000
6           minimum_distance      0.312
7            median_distance      2.374
8           maximum_distance     14.339


In [39]:
nearest_blocker_diagnostics = analytical_table[
    RUSHER_KEYS
    + [
        "rusher_position_lined_up",
        "original_play_direction",
        "rusher_x",
        "rusher_y",
        "quarterback_x",
        "quarterback_y",
    ]
].merge(
    nearest_blocker_reference,
    on=RUSHER_KEYS,
    how="left",
    validate="one_to_one",
)

assert len(nearest_blocker_diagnostics) == 36259
assert nearest_blocker_diagnostics.isna().sum().sum() == 0
assert "pressure" not in nearest_blocker_diagnostics.columns

distance_thresholds = [5.0, 7.5, 10.0]

nearest_distance_tail_summary = pd.DataFrame(
    [
        {
            "distance_above_yards": threshold,
            "rusher_rows": int(
                (
                    nearest_blocker_diagnostics[
                        "nearest_blocker_distance"
                    ]
                    > threshold
                ).sum()
            ),
            "rusher_rows_pct": round(
                (
                    nearest_blocker_diagnostics[
                        "nearest_blocker_distance"
                    ]
                    > threshold
                ).mean()
                * 100,
                3,
            ),
            "affected_plays": (
                nearest_blocker_diagnostics.loc[
                    nearest_blocker_diagnostics[
                        "nearest_blocker_distance"
                    ]
                    > threshold,
                    PLAY_KEYS,
                ]
                .drop_duplicates()
                .shape[0]
            ),
        }
        for threshold in distance_thresholds
    ]
)

largest_nearest_blocker_distances = (
    nearest_blocker_diagnostics[
        RUSHER_KEYS
        + [
            "rusher_position_lined_up",
            "original_play_direction",
            "rusher_x",
            "rusher_y",
            "quarterback_x",
            "quarterback_y",
            "nearest_blocker_nfl_id",
            "nearest_blocker_x",
            "nearest_blocker_y",
            "nearest_blocker_longitudinal_offset",
            "nearest_blocker_lateral_offset",
            "nearest_blocker_distance",
        ]
    ]
    .nlargest(
        10,
        "nearest_blocker_distance",
    )
    .reset_index(drop=True)
    .round(3)
)

tied_rusher_keys = (
    minimum_tie_counts.loc[
        minimum_tie_counts.gt(1)
    ]
    .reset_index()[RUSHER_KEYS]
)

tied_minimum_candidates = (
    rusher_blocker_candidates.loc[
        is_minimum_distance,
        PAIR_KEYS
        + [
            "rusher_x",
            "rusher_y",
            "blocker_x",
            "blocker_y",
            "rusher_blocker_longitudinal_offset",
            "rusher_blocker_lateral_offset",
            "rusher_blocker_distance",
        ],
    ]
    .merge(
        tied_rusher_keys,
        on=RUSHER_KEYS,
        how="inner",
        validate="many_to_one",
    )
    .sort_values(
        RUSHER_KEYS + ["pass_blocker_nfl_id"]
    )
    .reset_index(drop=True)
    .round(3)
)

assert len(tied_rusher_keys) == 1
assert len(tied_minimum_candidates) >= 2

display(nearest_distance_tail_summary)
display(largest_nearest_blocker_distances)
display(tied_minimum_candidates)

,distance_above_yards,rusher_rows,rusher_rows_pct,affected_plays
0,5.0,1034,2.852,938
1,7.5,164,0.452,162
2,10.0,18,0.050,18


,actual_week,gameId,playId,pass_rusher_nfl_id,rusher_position_lined_up,original_play_direction,rusher_x,rusher_y,quarterback_x,quarterback_y,nearest_blocker_nfl_id,nearest_blocker_x,nearest_blocker_y,nearest_blocker_longitudinal_offset,nearest_blocker_lateral_offset,nearest_blocker_distance
0,1,2021091201,4233,38548,ROLB,right,87.54,46.830,85.52,29.880,44875,89.86,32.680,2.32,-14.15,14.339
1,1,2021091200,2214,44893,RCB,right,83.60,44.800,74.60,29.740,46302,78.18,32.740,-5.42,-12.06,13.222
2,3,2021092604,2914,41502,SSR,right,75.49,38.130,60.33,29.790,53442,63.53,33.070,-11.96,-5.06,12.986
3,4,2021100312,2903,46168,LCB,right,73.60,8.810,67.45,23.820,43444,71.07,21.000,-2.53,12.19,12.450
4,4,2021100311,1983,35459,SCBR,left,106.85,36.223,91.48,29.573,41222,95.49,32.393,-11.36,-3.83,11.988
5,4,2021093000,276,53462,RCB,left,22.02,43.633,17.34,29.483,47794,17.61,32.553,-4.41,-11.08,11.925
6,1,2021090900,2530,46132,RCB,right,35.04,44.550,30.11,29.780,37082,33.41,33.070,-1.63,-11.48,11.595
7,1,2021091203,3429,44872,LCB,right,69.63,10.080,65.49,24.090,46131,65.70,20.920,-3.93,10.84,11.530
8,7,2021102403,2959,47828,RCB,left,76.24,43.893,73.70,29.613,43586,73.68,32.693,-2.56,-11.20,11.489
9,3,2021092611,1338,52607,SSL,right,59.62,19.170,44.45,23.970,40151,49.20,23.720,-10.42,4.55,11.370


,actual_week,gameId,playId,pass_rusher_nfl_id,pass_blocker_nfl_id,rusher_x,rusher_y,blocker_x,blocker_y,rusher_blocker_longitudinal_offset,rusher_blocker_lateral_offset,rusher_blocker_distance
0,5,2021101001,2019,38542,41436,60.0,22.9,58.52,24.12,-1.48,1.22,1.918
1,5,2021101001,2019,38542,42424,60.0,22.9,58.12,22.52,-1.88,-0.38,1.918


### Decision on nearest pass-blocker distance

Each pass rusher was paired with every `Pass Block` player in the same play
at the validated snap frame. The geometrically nearest blocker was then
selected using Euclidean distance.

Validation results:

- 196,361 candidate rusher-blocker pairs were evaluated.
- Every one of the 36,259 pass rushers received exactly one nearest-blocker
  reference.
- Median nearest-blocker distance: 2.374 yards.
- Minimum distance: 0.312 yards.
- Maximum distance: 14.339 yards.
- 1,034 observations (2.852%) exceed 5 yards.
- 164 observations (0.452%) exceed 7.5 yards.
- 18 observations (0.050%) exceed 10 yards.

The largest distances primarily correspond to defensive backs, safeties,
and outside linebackers aligned away from the offensive line. These
positions can participate in a blitz and therefore may legitimately be far
from every pass blocker at the snap.

One pass rusher had two blockers at the same minimum distance. The tie was
resolved deterministically using the smallest blocker `nflId`. This
identifier is retained only for traceability and will not be used as a
model predictor.

Project decision:

1. Retain all nearest-blocker distances without clipping or row removal.
2. Treat the nearest blocker as a geometric snap-time proxy, not as the
   confirmed player responsible for the block.
3. Do not use `pff_nflIdBlockedPlayer` or other post-snap scouting outcomes
   to determine the matchup.
4. Preserve signed longitudinal and lateral offsets.
5. Exclude blocker and rusher identifiers from the predictor matrix.
6. Preserve all 36,259 analytical observations.

In [40]:
analytical_table_before_blocker = analytical_table.copy()

analytical_table = analytical_table.merge(
    nearest_blocker_reference,
    on=RUSHER_KEYS,
    how="left",
    validate="one_to_one",
)

analytical_table[
    "nearest_blocker_absolute_lateral_offset"
] = analytical_table[
    "nearest_blocker_lateral_offset"
].abs()

NEAREST_BLOCKER_FEATURE_COLUMNS = [
    "nearest_blocker_x",
    "nearest_blocker_y",
    "nearest_blocker_speed",
    "nearest_blocker_acceleration",
    "nearest_blocker_displacement",
    "nearest_blocker_orientation",
    "nearest_blocker_direction",
    "nearest_blocker_longitudinal_offset",
    "nearest_blocker_lateral_offset",
    "nearest_blocker_absolute_lateral_offset",
    "nearest_blocker_distance",
]

assert len(analytical_table) == 36259
assert analytical_table.shape[1] == 41
assert not analytical_table.duplicated(RUSHER_KEYS).any()
assert analytical_table[
    NEAREST_BLOCKER_FEATURE_COLUMNS
].notna().all().all()
assert np.isfinite(
    analytical_table[
        NEAREST_BLOCKER_FEATURE_COLUMNS
    ].to_numpy()
).all()
assert analytical_table["nearest_blocker_nfl_id"].notna().all()
assert not excluded_source_columns.intersection(
    analytical_table.columns
)

np.testing.assert_allclose(
    analytical_table[
        "nearest_blocker_longitudinal_offset"
    ],
    analytical_table["nearest_blocker_x"]
    - analytical_table["rusher_x"],
    atol=1e-12,
)

np.testing.assert_allclose(
    analytical_table["nearest_blocker_lateral_offset"],
    analytical_table["nearest_blocker_y"]
    - analytical_table["rusher_y"],
    atol=1e-12,
)

np.testing.assert_allclose(
    analytical_table["nearest_blocker_distance"],
    np.hypot(
        analytical_table[
            "nearest_blocker_longitudinal_offset"
        ],
        analytical_table[
            "nearest_blocker_lateral_offset"
        ],
    ),
    atol=1e-12,
)

pressure_before_blocker = (
    analytical_table_before_blocker.set_index(RUSHER_KEYS)[
        "pressure"
    ]
    .sort_index()
)

pressure_after_blocker = (
    analytical_table.set_index(RUSHER_KEYS)["pressure"]
    .sort_index()
)

pd.testing.assert_series_equal(
    pressure_before_blocker,
    pressure_after_blocker,
)

nearest_blocker_feature_quality = pd.DataFrame(
    [
        {
            "feature": feature,
            "missing": analytical_table[feature].isna().sum(),
            "non_finite": (
                ~np.isfinite(analytical_table[feature])
            ).sum(),
            "minimum": analytical_table[feature].min(),
            "median": analytical_table[feature].median(),
            "maximum": analytical_table[feature].max(),
        }
        for feature in NEAREST_BLOCKER_FEATURE_COLUMNS
    ]
).round(3)

analytical_table_blocker_summary = pd.DataFrame(
    [
        {
            "metric": "analytical_rows",
            "value": len(analytical_table),
        },
        {
            "metric": "columns",
            "value": analytical_table.shape[1],
        },
        {
            "metric": "unique_rusher_keys",
            "value": analytical_table[
                RUSHER_KEYS
            ].drop_duplicates().shape[0],
        },
        {
            "metric": "modeling_plays",
            "value": analytical_table[
                PLAY_KEYS
            ].drop_duplicates().shape[0],
        },
        {
            "metric": "missing_values",
            "value": analytical_table.isna().sum().sum(),
        },
        {
            "metric": "pressure_positives",
            "value": int(
                analytical_table["pressure"].sum()
            ),
        },
        {
            "metric": "nearest_blocker_features",
            "value": len(
                NEAREST_BLOCKER_FEATURE_COLUMNS
            ),
        },
    ]
)

display(analytical_table_blocker_summary)
display(nearest_blocker_feature_quality)

,metric,value
0,analytical_rows,36259
1,columns,41
2,unique_rusher_keys,36259
3,modeling_plays,8531
4,missing_values,0
5,pressure_positives,4214
6,nearest_blocker_features,11


,feature,missing,non_finite,minimum,median,maximum
0,nearest_blocker_x,0,0,8.830,54.170,109.360
1,nearest_blocker_y,0,0,15.150,26.700,36.640
2,nearest_blocker_speed,0,0,0.000,0.180,3.300
3,nearest_blocker_acceleration,0,0,0.000,0.850,5.370
4,nearest_blocker_displacement,0,0,0.000,0.020,0.360
5,nearest_blocker_orientation,0,0,0.040,88.040,359.710
6,nearest_blocker_direction,0,0,0.030,260.700,359.930
7,nearest_blocker_longitudinal_offset,0,0,-11.960,-1.960,2.320
8,nearest_blocker_lateral_offset,0,0,-14.150,-0.020,12.190
9,nearest_blocker_absolute_lateral_offset,0,0,0.000,1.120,14.150


In [41]:
analytical_table_before_context = analytical_table.copy()

play_protection_context = (
    play_entity_counts[
        PLAY_KEYS
        + [
            "pass_rushers",
            "pass_blockers",
        ]
    ]
    .rename(
        columns={
            "pass_rushers": "play_pass_rusher_count",
            "pass_blockers": "play_pass_blocker_count",
        }
    )
    .copy()
)

play_protection_context[
    "play_pass_rusher_count"
] = play_protection_context[
    "play_pass_rusher_count"
].astype("int16")

play_protection_context[
    "play_pass_blocker_count"
] = play_protection_context[
    "play_pass_blocker_count"
].astype("int16")

play_protection_context[
    "protection_count_advantage"
] = (
    play_protection_context["play_pass_blocker_count"]
    - play_protection_context["play_pass_rusher_count"]
).astype("int16")

PLAY_CONTEXT_FEATURE_COLUMNS = [
    "play_pass_rusher_count",
    "play_pass_blocker_count",
    "protection_count_advantage",
]

assert len(play_protection_context) == 8531
assert not play_protection_context.duplicated(PLAY_KEYS).any()
assert play_protection_context[
    PLAY_CONTEXT_FEATURE_COLUMNS
].notna().all().all()

assert int(
    play_protection_context[
        "play_pass_rusher_count"
    ].sum()
) == 36259

assert int(
    play_protection_context[
        "play_pass_blocker_count"
    ].sum()
) == 45921

analytical_table = analytical_table.merge(
    play_protection_context,
    on=PLAY_KEYS,
    how="left",
    validate="many_to_one",
)

assert len(analytical_table) == 36259
assert analytical_table.shape[1] == 44
assert not analytical_table.duplicated(RUSHER_KEYS).any()
assert analytical_table[
    PLAY_CONTEXT_FEATURE_COLUMNS
].notna().all().all()
assert np.isfinite(
    analytical_table[
        PLAY_CONTEXT_FEATURE_COLUMNS
    ].to_numpy()
).all()
assert not excluded_source_columns.intersection(
    analytical_table.columns
)

pressure_before_context = (
    analytical_table_before_context.set_index(RUSHER_KEYS)[
        "pressure"
    ]
    .sort_index()
)

pressure_after_context = (
    analytical_table.set_index(RUSHER_KEYS)["pressure"]
    .sort_index()
)

pd.testing.assert_series_equal(
    pressure_before_context,
    pressure_after_context,
)

play_context_quality = pd.DataFrame(
    [
        {
            "feature": feature,
            "unit": "play",
            "missing": (
                play_protection_context[feature]
                .isna()
                .sum()
            ),
            "minimum": (
                play_protection_context[feature].min()
            ),
            "median": (
                play_protection_context[feature].median()
            ),
            "maximum": (
                play_protection_context[feature].max()
            ),
            "unique_values": (
                play_protection_context[feature].nunique()
            ),
        }
        for feature in PLAY_CONTEXT_FEATURE_COLUMNS
    ]
)

analytical_table_context_summary = pd.DataFrame(
    [
        {
            "metric": "analytical_rows",
            "value": len(analytical_table),
        },
        {
            "metric": "columns",
            "value": analytical_table.shape[1],
        },
        {
            "metric": "modeling_plays",
            "value": analytical_table[
                PLAY_KEYS
            ].drop_duplicates().shape[0],
        },
        {
            "metric": "missing_values",
            "value": analytical_table.isna().sum().sum(),
        },
        {
            "metric": "pressure_positives",
            "value": int(
                analytical_table["pressure"].sum()
            ),
        },
        {
            "metric": "play_context_features",
            "value": len(
                PLAY_CONTEXT_FEATURE_COLUMNS
            ),
        },
    ]
)

display(analytical_table_context_summary)
display(play_context_quality)

,metric,value
0,analytical_rows,36259
1,columns,44
2,modeling_plays,8531
3,missing_values,0
4,pressure_positives,4214
5,play_context_features,3


,feature,unit,missing,minimum,median,maximum,unique_values
0,play_pass_rusher_count,play,0,1,4.0,8,8
1,play_pass_blocker_count,play,0,5,5.0,9,5
2,protection_count_advantage,play,0,-2,1.0,6,9


In [42]:
analytical_table_before_angles = analytical_table.copy()

ANGLE_COLUMNS = [
    "rusher_orientation",
    "rusher_direction",
    "quarterback_orientation",
    "quarterback_direction",
    "nearest_blocker_orientation",
    "nearest_blocker_direction",
]

CYCLICAL_ANGLE_FEATURE_COLUMNS = []

for angle_column in ANGLE_COLUMNS:
    angle_radians = np.deg2rad(
        analytical_table[angle_column].mod(
            FULL_TURN_DEGREES
        )
    )

    sine_column = f"{angle_column}_sin"
    cosine_column = f"{angle_column}_cos"

    analytical_table[sine_column] = np.sin(
        angle_radians
    )
    analytical_table[cosine_column] = np.cos(
        angle_radians
    )

    CYCLICAL_ANGLE_FEATURE_COLUMNS.extend(
        [
            sine_column,
            cosine_column,
        ]
    )

    np.testing.assert_allclose(
        (
            analytical_table[sine_column] ** 2
            + analytical_table[cosine_column] ** 2
        ),
        1.0,
        atol=1e-12,
    )

assert len(analytical_table) == 36259
assert analytical_table.shape[1] == 56
assert not analytical_table.duplicated(RUSHER_KEYS).any()
assert analytical_table[
    CYCLICAL_ANGLE_FEATURE_COLUMNS
].notna().all().all()
assert np.isfinite(
    analytical_table[
        CYCLICAL_ANGLE_FEATURE_COLUMNS
    ].to_numpy()
).all()

assert (
    analytical_table[
        CYCLICAL_ANGLE_FEATURE_COLUMNS
    ]
    .ge(-1.0)
    .all()
    .all()
)

assert (
    analytical_table[
        CYCLICAL_ANGLE_FEATURE_COLUMNS
    ]
    .le(1.0)
    .all()
    .all()
)

pressure_before_angles = (
    analytical_table_before_angles.set_index(RUSHER_KEYS)[
        "pressure"
    ]
    .sort_index()
)

pressure_after_angles = (
    analytical_table.set_index(RUSHER_KEYS)["pressure"]
    .sort_index()
)

pd.testing.assert_series_equal(
    pressure_before_angles,
    pressure_after_angles,
)

cyclical_angle_quality = pd.DataFrame(
    [
        {
            "source_angle": angle_column,
            "sine_minimum": analytical_table[
                f"{angle_column}_sin"
            ].min(),
            "sine_maximum": analytical_table[
                f"{angle_column}_sin"
            ].max(),
            "cosine_minimum": analytical_table[
                f"{angle_column}_cos"
            ].min(),
            "cosine_maximum": analytical_table[
                f"{angle_column}_cos"
            ].max(),
            "maximum_unit_circle_error": (
                (
                    analytical_table[
                        f"{angle_column}_sin"
                    ]
                    ** 2
                    + analytical_table[
                        f"{angle_column}_cos"
                    ]
                    ** 2
                    - 1.0
                )
                .abs()
                .max()
            ),
        }
        for angle_column in ANGLE_COLUMNS
    ]
)

analytical_table_angle_summary = pd.DataFrame(
    [
        {
            "metric": "analytical_rows",
            "value": len(analytical_table),
        },
        {
            "metric": "columns",
            "value": analytical_table.shape[1],
        },
        {
            "metric": "cyclical_angle_features",
            "value": len(
                CYCLICAL_ANGLE_FEATURE_COLUMNS
            ),
        },
        {
            "metric": "missing_angle_features",
            "value": analytical_table[
                CYCLICAL_ANGLE_FEATURE_COLUMNS
            ].isna().sum().sum(),
        },
        {
            "metric": "pressure_positives",
            "value": int(
                analytical_table["pressure"].sum()
            ),
        },
    ]
)

display(analytical_table_angle_summary)
display(cyclical_angle_quality)

,metric,value
0,analytical_rows,36259
1,columns,56
2,cyclical_angle_features,12
3,missing_angle_features,0
4,pressure_positives,4214


,source_angle,sine_minimum,sine_maximum,cosine_minimum,cosine_maximum,maximum_unit_circle_error
0,rusher_orientation,-1.000000,1.0,-1.000000,1.000000,2.220446e-16
1,rusher_direction,-1.000000,1.0,-1.000000,1.000000,2.220446e-16
2,quarterback_orientation,-0.999835,1.0,-0.999910,0.999992,2.220446e-16
3,quarterback_direction,-1.000000,1.0,-0.999994,0.999999,2.220446e-16
4,nearest_blocker_orientation,-1.000000,1.0,-0.999998,1.000000,2.220446e-16
5,nearest_blocker_direction,-1.000000,1.0,-1.000000,1.000000,2.220446e-16


In [43]:
MINIMUM_MOVEMENT_SPEED = 0.5

rusher_to_qb_unit_x = (
    analytical_table["quarterback_x"]
    - analytical_table["rusher_x"]
) / analytical_table["rusher_qb_distance"]

rusher_to_qb_unit_y = (
    analytical_table["quarterback_y"]
    - analytical_table["rusher_y"]
) / analytical_table["rusher_qb_distance"]

blocker_to_rusher_unit_x = (
    analytical_table["rusher_x"]
    - analytical_table["nearest_blocker_x"]
) / analytical_table["nearest_blocker_distance"]

blocker_to_rusher_unit_y = (
    analytical_table["rusher_y"]
    - analytical_table["nearest_blocker_y"]
) / analytical_table["nearest_blocker_distance"]

rusher_moving_mask = (
    analytical_table["rusher_speed"]
    >= MINIMUM_MOVEMENT_SPEED
)

blocker_moving_mask = (
    analytical_table["nearest_blocker_speed"]
    >= MINIMUM_MOVEMENT_SPEED
)

angle_conventions = {
    "zero_degrees_toward_negative_y": -1.0,
    "zero_degrees_toward_positive_y": 1.0,
}

alignment_results = []

for convention_name, y_component_sign in (
    angle_conventions.items()
):
    alignment_measurements = {
        "rusher_orientation_to_qb": (
            analytical_table["rusher_orientation_sin"]
            * rusher_to_qb_unit_x
            + y_component_sign
            * analytical_table["rusher_orientation_cos"]
            * rusher_to_qb_unit_y,
            pd.Series(True, index=analytical_table.index),
        ),
        "rusher_direction_to_qb": (
            analytical_table["rusher_direction_sin"]
            * rusher_to_qb_unit_x
            + y_component_sign
            * analytical_table["rusher_direction_cos"]
            * rusher_to_qb_unit_y,
            rusher_moving_mask,
        ),
        "blocker_orientation_to_rusher": (
            analytical_table[
                "nearest_blocker_orientation_sin"
            ]
            * blocker_to_rusher_unit_x
            + y_component_sign
            * analytical_table[
                "nearest_blocker_orientation_cos"
            ]
            * blocker_to_rusher_unit_y,
            pd.Series(True, index=analytical_table.index),
        ),
        "blocker_direction_to_rusher": (
            analytical_table[
                "nearest_blocker_direction_sin"
            ]
            * blocker_to_rusher_unit_x
            + y_component_sign
            * analytical_table[
                "nearest_blocker_direction_cos"
            ]
            * blocker_to_rusher_unit_y,
            blocker_moving_mask,
        ),
    }

    for measurement_name, (
        alignment_values,
        valid_mask,
    ) in alignment_measurements.items():
        selected_values = alignment_values.loc[
            valid_mask
        ]

        assert selected_values.notna().all()
        assert selected_values.between(
            -1.0 - 1e-12,
            1.0 + 1e-12,
        ).all()

        alignment_results.append(
            {
                "angle_convention": convention_name,
                "measurement": measurement_name,
                "rows_evaluated": len(selected_values),
                "mean_alignment": selected_values.mean(),
                "median_alignment": selected_values.median(),
                "positive_alignment_pct": (
                    selected_values.gt(0.0).mean()
                    * 100
                ),
            }
        )

angle_convention_diagnostics = pd.DataFrame(
    alignment_results
).round(3)

assert "pressure" not in angle_convention_diagnostics.columns

print(angle_convention_diagnostics)

                 angle_convention                    measurement  \
0  zero_degrees_toward_negative_y       rusher_orientation_to_qb   
1  zero_degrees_toward_negative_y         rusher_direction_to_qb   
2  zero_degrees_toward_negative_y  blocker_orientation_to_rusher   
3  zero_degrees_toward_negative_y    blocker_direction_to_rusher   
4  zero_degrees_toward_positive_y       rusher_orientation_to_qb   
5  zero_degrees_toward_positive_y         rusher_direction_to_qb   
6  zero_degrees_toward_positive_y  blocker_orientation_to_rusher   
7  zero_degrees_toward_positive_y    blocker_direction_to_rusher   

   rows_evaluated  mean_alignment  median_alignment  positive_alignment_pct  
0           36259           0.549             0.699                  86.527  
1            7194           0.586             0.716                  88.866  
2           36259           0.726             0.820                  97.002  
3            5231          -0.691            -0.871                   9.826

### Empirical validation of the angular axis convention

The tracking schema expresses player orientation and movement direction as
angles between 0 and 360 degrees. Before projecting those angles onto the
normalized field axes, both possible signs of the vertical component were
tested without using the target variable.

The comparison evaluated:

- Pass-rusher body orientation toward the quarterback.
- Pass-rusher movement direction toward the quarterback.
- Nearest-blocker body orientation toward the pass rusher.
- Nearest-blocker movement direction toward the pass rusher.

The convention where zero degrees points toward increasing normalized `y`
showed the strongest and most consistent geometric agreement:

- Median rusher-orientation alignment: `0.891`.
- Positive rusher-orientation alignment: `95.273%`.
- Median moving-rusher direction alignment: `0.836`.
- Positive moving-rusher direction alignment: `96.038%`.
- Median nearest-blocker orientation alignment: `0.831`.

Blocker movement direction was often opposite the pass rusher under both
conventions. This is plausible because pass blockers frequently move
backward at the snap to establish pass protection while keeping their body
oriented toward the approaching defender.

Project decision:

1. Use the empirically supported angular projection:

   - `unit_x = sin(angle)`
   - `unit_y = cos(angle)`

2. Apply it only to the already normalized orientation and direction angles.
3. Use movement direction only together with player speed.
4. Retain body orientation and movement direction as distinct concepts.
5. Do not use `pressure` or any outcome component to select the convention.
6. Preserve raw angles for audit, but use cyclical and derived movement
   features in the future predictor matrix.

In [44]:
analytical_table_before_movement = analytical_table.copy()

VELOCITY_COMPONENT_SPECS = [
    (
        "rusher",
        "rusher_speed",
        "rusher_direction_sin",
        "rusher_direction_cos",
    ),
    (
        "quarterback",
        "quarterback_speed",
        "quarterback_direction_sin",
        "quarterback_direction_cos",
    ),
    (
        "nearest_blocker",
        "nearest_blocker_speed",
        "nearest_blocker_direction_sin",
        "nearest_blocker_direction_cos",
    ),
]

for (
    entity_prefix,
    speed_column,
    direction_sine_column,
    direction_cosine_column,
) in VELOCITY_COMPONENT_SPECS:
    analytical_table[
        f"{entity_prefix}_velocity_x"
    ] = (
        analytical_table[speed_column]
        * analytical_table[direction_sine_column]
    )

    analytical_table[
        f"{entity_prefix}_velocity_y"
    ] = (
        analytical_table[speed_column]
        * analytical_table[direction_cosine_column]
    )

# Unit vector from the pass rusher toward the quarterback.
rusher_to_qb_unit_x = (
    analytical_table["quarterback_x"]
    - analytical_table["rusher_x"]
) / analytical_table["rusher_qb_distance"]

rusher_to_qb_unit_y = (
    analytical_table["quarterback_y"]
    - analytical_table["rusher_y"]
) / analytical_table["rusher_qb_distance"]

# Unit vector from the nearest blocker toward the pass rusher.
blocker_to_rusher_unit_x = (
    analytical_table["rusher_x"]
    - analytical_table["nearest_blocker_x"]
) / analytical_table["nearest_blocker_distance"]

blocker_to_rusher_unit_y = (
    analytical_table["rusher_y"]
    - analytical_table["nearest_blocker_y"]
) / analytical_table["nearest_blocker_distance"]

analytical_table[
    "rusher_orientation_to_qb_alignment"
] = (
    analytical_table["rusher_orientation_sin"]
    * rusher_to_qb_unit_x
    + analytical_table["rusher_orientation_cos"]
    * rusher_to_qb_unit_y
)

analytical_table[
    "rusher_direction_to_qb_alignment"
] = (
    analytical_table["rusher_direction_sin"]
    * rusher_to_qb_unit_x
    + analytical_table["rusher_direction_cos"]
    * rusher_to_qb_unit_y
)

analytical_table[
    "rusher_velocity_toward_qb"
] = (
    analytical_table["rusher_velocity_x"]
    * rusher_to_qb_unit_x
    + analytical_table["rusher_velocity_y"]
    * rusher_to_qb_unit_y
)

analytical_table[
    "rusher_qb_closing_speed"
] = (
    (
        analytical_table["rusher_velocity_x"]
        - analytical_table["quarterback_velocity_x"]
    )
    * rusher_to_qb_unit_x
    + (
        analytical_table["rusher_velocity_y"]
        - analytical_table["quarterback_velocity_y"]
    )
    * rusher_to_qb_unit_y
)

analytical_table[
    "nearest_blocker_orientation_to_rusher_alignment"
] = (
    analytical_table[
        "nearest_blocker_orientation_sin"
    ]
    * blocker_to_rusher_unit_x
    + analytical_table[
        "nearest_blocker_orientation_cos"
    ]
    * blocker_to_rusher_unit_y
)

analytical_table[
    "nearest_blocker_direction_to_rusher_alignment"
] = (
    analytical_table[
        "nearest_blocker_direction_sin"
    ]
    * blocker_to_rusher_unit_x
    + analytical_table[
        "nearest_blocker_direction_cos"
    ]
    * blocker_to_rusher_unit_y
)

analytical_table[
    "nearest_blocker_velocity_toward_rusher"
] = (
    analytical_table["nearest_blocker_velocity_x"]
    * blocker_to_rusher_unit_x
    + analytical_table["nearest_blocker_velocity_y"]
    * blocker_to_rusher_unit_y
)

analytical_table[
    "rusher_blocker_closing_speed"
] = (
    (
        analytical_table["nearest_blocker_velocity_x"]
        - analytical_table["rusher_velocity_x"]
    )
    * blocker_to_rusher_unit_x
    + (
        analytical_table["nearest_blocker_velocity_y"]
        - analytical_table["rusher_velocity_y"]
    )
    * blocker_to_rusher_unit_y
)

MOVEMENT_FEATURE_COLUMNS = [
    "rusher_velocity_x",
    "rusher_velocity_y",
    "quarterback_velocity_x",
    "quarterback_velocity_y",
    "nearest_blocker_velocity_x",
    "nearest_blocker_velocity_y",
    "rusher_orientation_to_qb_alignment",
    "rusher_direction_to_qb_alignment",
    "rusher_velocity_toward_qb",
    "rusher_qb_closing_speed",
    "nearest_blocker_orientation_to_rusher_alignment",
    "nearest_blocker_direction_to_rusher_alignment",
    "nearest_blocker_velocity_toward_rusher",
    "rusher_blocker_closing_speed",
]

ALIGNMENT_FEATURE_COLUMNS = [
    "rusher_orientation_to_qb_alignment",
    "rusher_direction_to_qb_alignment",
    "nearest_blocker_orientation_to_rusher_alignment",
    "nearest_blocker_direction_to_rusher_alignment",
]

assert len(analytical_table) == 36259
assert analytical_table.shape[1] == 70
assert not analytical_table.duplicated(RUSHER_KEYS).any()
assert analytical_table[
    MOVEMENT_FEATURE_COLUMNS
].notna().all().all()
assert np.isfinite(
    analytical_table[MOVEMENT_FEATURE_COLUMNS].to_numpy()
).all()

assert (
    analytical_table[ALIGNMENT_FEATURE_COLUMNS]
    .ge(-1.0 - 1e-12)
    .all()
    .all()
)
assert (
    analytical_table[ALIGNMENT_FEATURE_COLUMNS]
    .le(1.0 + 1e-12)
    .all()
    .all()
)

for (
    entity_prefix,
    speed_column,
    _,
    _,
) in VELOCITY_COMPONENT_SPECS:
    np.testing.assert_allclose(
        np.hypot(
            analytical_table[
                f"{entity_prefix}_velocity_x"
            ],
            analytical_table[
                f"{entity_prefix}_velocity_y"
            ],
        ),
        analytical_table[speed_column],
        atol=1e-12,
    )

assert (
    analytical_table[
        "rusher_velocity_toward_qb"
    ].abs()
    <= analytical_table["rusher_speed"] + 1e-12
).all()

assert (
    analytical_table[
        "rusher_qb_closing_speed"
    ].abs()
    <= (
        analytical_table["rusher_speed"]
        + analytical_table["quarterback_speed"]
        + 1e-12
    )
).all()

assert (
    analytical_table[
        "rusher_blocker_closing_speed"
    ].abs()
    <= (
        analytical_table["rusher_speed"]
        + analytical_table["nearest_blocker_speed"]
        + 1e-12
    )
).all()

pressure_before_movement = (
    analytical_table_before_movement.set_index(RUSHER_KEYS)[
        "pressure"
    ]
    .sort_index()
)
pressure_after_movement = (
    analytical_table.set_index(RUSHER_KEYS)["pressure"]
    .sort_index()
)

pd.testing.assert_series_equal(
    pressure_before_movement,
    pressure_after_movement,
)

movement_feature_units = {
    feature: (
        "unitless"
        if feature in ALIGNMENT_FEATURE_COLUMNS
        else "yards_per_second"
    )
    for feature in MOVEMENT_FEATURE_COLUMNS
}

movement_feature_quality = pd.DataFrame(
    [
        {
            "feature": feature,
            "unit": movement_feature_units[feature],
            "missing": analytical_table[feature].isna().sum(),
            "non_finite": (
                ~np.isfinite(analytical_table[feature])
            ).sum(),
            "minimum": analytical_table[feature].min(),
            "median": analytical_table[feature].median(),
            "maximum": analytical_table[feature].max(),
        }
        for feature in MOVEMENT_FEATURE_COLUMNS
    ]
).round(3)

analytical_table_movement_summary = pd.DataFrame(
    [
        {
            "metric": "analytical_rows",
            "value": len(analytical_table),
        },
        {
            "metric": "columns",
            "value": analytical_table.shape[1],
        },
        {
            "metric": "movement_features",
            "value": len(MOVEMENT_FEATURE_COLUMNS),
        },
        {
            "metric": "missing_values",
            "value": analytical_table.isna().sum().sum(),
        },
        {
            "metric": "pressure_positives",
            "value": int(
                analytical_table["pressure"].sum()
            ),
        },
    ]
)

display(analytical_table_movement_summary)
display(movement_feature_quality)

,metric,value
0,analytical_rows,36259
1,columns,70
2,movement_features,14
3,missing_values,0
4,pressure_positives,4214


,feature,unit,missing,non_finite,minimum,median,maximum
0,rusher_velocity_x,yards_per_second,0,0,-5.437,-0.166,1.991
1,rusher_velocity_y,yards_per_second,0,0,-6.005,0.000,4.619
2,quarterback_velocity_x,yards_per_second,0,0,-4.180,0.000,0.754
3,quarterback_velocity_y,yards_per_second,0,0,-0.686,0.000,0.826
4,nearest_blocker_velocity_x,yards_per_second,0,0,-3.295,-0.094,0.913
5,nearest_blocker_velocity_y,yards_per_second,0,0,-1.917,0.000,1.796
6,rusher_orientation_to_qb_alignment,unitless,0,0,-1.000,0.891,1.000
7,rusher_direction_to_qb_alignment,unitless,0,0,-1.000,0.791,1.000
8,rusher_velocity_toward_qb,yards_per_second,0,0,-2.293,0.139,4.925
9,rusher_qb_closing_speed,yards_per_second,0,0,-4.020,0.122,4.862


In [45]:
IDENTIFIER_METADATA_COLUMNS = [
    "gameId",
    "playId",
    "pass_rusher_nfl_id",
    "quarterback_nfl_id",
    "nearest_blocker_nfl_id",
]

SPLIT_METADATA_COLUMNS = [
    "actual_week",
]

AUDIT_ONLY_COLUMNS = [
    "original_play_direction",
    "rusher_x",
    "rusher_y",
    "rusher_orientation",
    "rusher_direction",
    "quarterback_x",
    "quarterback_orientation",
    "quarterback_direction",
    "nearest_blocker_x",
    "nearest_blocker_y",
    "nearest_blocker_orientation",
    "nearest_blocker_direction",
]

TARGET_COLUMNS = [
    "pressure",
]

CATEGORICAL_PREDICTOR_COLUMNS = [
    "rusher_position_lined_up",
]

NUMERIC_PREDICTOR_COLUMNS = [
    # Spatial anchors.
    "football_x",
    "quarterback_y",

    # Player kinematics at the snap.
    "rusher_speed",
    "rusher_acceleration",
    "rusher_displacement",
    "quarterback_speed",
    "quarterback_acceleration",
    "quarterback_displacement",

    # Pass-rusher relationships.
    *SPATIAL_FEATURE_COLUMNS,

    # Nearest-blocker kinematics and relationships.
    "nearest_blocker_speed",
    "nearest_blocker_acceleration",
    "nearest_blocker_displacement",
    "nearest_blocker_longitudinal_offset",
    "nearest_blocker_lateral_offset",
    "nearest_blocker_absolute_lateral_offset",
    "nearest_blocker_distance",

    # Play-level numerical context.
    *PLAY_CONTEXT_FEATURE_COLUMNS,

    # Cyclical representations of angles.
    *CYCLICAL_ANGLE_FEATURE_COLUMNS,

    # Velocity, alignment, and closing-speed features.
    *MOVEMENT_FEATURE_COLUMNS,
]

PREDICTOR_COLUMNS = (
    CATEGORICAL_PREDICTOR_COLUMNS
    + NUMERIC_PREDICTOR_COLUMNS
)

COLUMN_ROLE_GROUPS = {
    "identifier_metadata": IDENTIFIER_METADATA_COLUMNS,
    "split_metadata": SPLIT_METADATA_COLUMNS,
    "audit_only": AUDIT_ONLY_COLUMNS,
    "target": TARGET_COLUMNS,
    "categorical_predictor": (
        CATEGORICAL_PREDICTOR_COLUMNS
    ),
    "numeric_predictor": NUMERIC_PREDICTOR_COLUMNS,
}

assigned_columns = [
    column
    for columns in COLUMN_ROLE_GROUPS.values()
    for column in columns
]

assignment_counts = pd.Series(
    assigned_columns
).value_counts()

duplicated_assignments = assignment_counts.loc[
    assignment_counts.gt(1)
].index.tolist()

unassigned_columns = sorted(
    set(analytical_table.columns)
    - set(assigned_columns)
)

unexpected_columns = sorted(
    set(assigned_columns)
    - set(analytical_table.columns)
)

FORBIDDEN_MODEL_COLUMNS = (
    set(IDENTIFIER_METADATA_COLUMNS)
    | set(SPLIT_METADATA_COLUMNS)
    | set(AUDIT_ONLY_COLUMNS)
    | set(TARGET_COLUMNS)
    | excluded_source_columns
    | {
        "has_valid_manual_snap",
        "football_y",
        "football_y_norm",
    }
)

assert analytical_table.columns.is_unique
assert not duplicated_assignments
assert not unassigned_columns
assert not unexpected_columns
assert len(assigned_columns) == 70
assert len(NUMERIC_PREDICTOR_COLUMNS) == 50
assert len(CATEGORICAL_PREDICTOR_COLUMNS) == 1
assert len(PREDICTOR_COLUMNS) == 51
assert set(PREDICTOR_COLUMNS).isdisjoint(
    FORBIDDEN_MODEL_COLUMNS
)
assert all(
    pd.api.types.is_numeric_dtype(
        analytical_table[column]
    )
    for column in NUMERIC_PREDICTOR_COLUMNS
)
assert analytical_table[
    CATEGORICAL_PREDICTOR_COLUMNS
].notna().all().all()
assert analytical_table[TARGET_COLUMNS].notna().all().all()
assert int(analytical_table["pressure"].sum()) == 4214
assert not excluded_source_columns.intersection(
    analytical_table.columns
)

column_inventory_records = []

for column_role, columns in COLUMN_ROLE_GROUPS.items():
    for column in columns:
        column_inventory_records.append(
            {
                "column": column,
                "column_role": column_role,
                "dtype": str(
                    analytical_table[column].dtype
                ),
                "missing": int(
                    analytical_table[column]
                    .isna()
                    .sum()
                ),
                "unique_values": int(
                    analytical_table[column].nunique()
                ),
                "model_eligible": column_role in {
                    "categorical_predictor",
                    "numeric_predictor",
                },
            }
        )

column_inventory = pd.DataFrame(
    column_inventory_records
)

column_role_summary = (
    column_inventory.groupby(
        "column_role",
        sort=False,
    )
    .agg(
        columns=("column", "size"),
        missing_values=("missing", "sum"),
        model_eligible_columns=(
            "model_eligible",
            "sum",
        ),
    )
    .reset_index()
)

assert len(column_inventory) == 70
assert column_inventory["missing"].sum() == 0
assert column_inventory[
    "model_eligible"
].sum() == 51

display(column_role_summary)
display(column_inventory)

,column_role,columns,missing_values,model_eligible_columns
0,identifier_metadata,5,0,0
1,split_metadata,1,0,0
2,audit_only,12,0,0
3,target,1,0,0
4,categorical_predictor,1,0,1
5,numeric_predictor,50,0,50


,column,column_role,dtype,missing,unique_values,model_eligible
0,gameId,identifier_metadata,int64,0,122,False
1,playId,identifier_metadata,int64,0,3754,False
2,pass_rusher_nfl_id,identifier_metadata,int64,0,698,False
3,quarterback_nfl_id,identifier_metadata,int64,0,60,False
4,nearest_blocker_nfl_id,identifier_metadata,int64,0,399,False
...,...,...,...,...,...,...
65,rusher_qb_closing_speed,numeric_predictor,float64,0,35484,True
66,nearest_blocker_orientation_to_rusher_alignment,numeric_predictor,float64,0,36255,True
67,nearest_blocker_direction_to_rusher_alignment,numeric_predictor,float64,0,36258,True
68,nearest_blocker_velocity_toward_rusher,numeric_predictor,float64,0,31945,True


In [47]:
expected_target_reference = (
    pressure_labels[
        [
            "actual_week",
            "gameId",
            "playId",
            "nflId",
            "pff_positionLinedUp",
            "pressure",
        ]
    ]
    .rename(
        columns={
            "nflId": "pass_rusher_nfl_id",
            "pff_positionLinedUp": (
                "rusher_position_lined_up"
            ),
        }
    )
    .sort_values(RUSHER_KEYS)
    .reset_index(drop=True)
)

actual_target_reference = (
    analytical_table[
        RUSHER_KEYS
        + [
            "rusher_position_lined_up",
            "pressure",
        ]
    ]
    .sort_values(RUSHER_KEYS)
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    expected_target_reference,
    actual_target_reference,
    check_dtype=False,
)

expected_week_integrity = (
    expected_target_reference.groupby(
        "actual_week",
        as_index=False,
    )
    .agg(
        pass_rusher_rows=(
            "pass_rusher_nfl_id",
            "size",
        ),
        pressure_positives=(
            "pressure",
            "sum",
        ),
    )
)

actual_week_integrity = (
    actual_target_reference.groupby(
        "actual_week",
        as_index=False,
    )
    .agg(
        pass_rusher_rows=(
            "pass_rusher_nfl_id",
            "size",
        ),
        pressure_positives=(
            "pressure",
            "sum",
        ),
    )
)

pd.testing.assert_frame_equal(
    expected_week_integrity,
    actual_week_integrity,
    check_dtype=False,
)

numeric_predictor_values = analytical_table[
    NUMERIC_PREDICTOR_COLUMNS
].to_numpy(dtype="float64")

constant_predictor_columns = [
    column
    for column in PREDICTOR_COLUMNS
    if analytical_table[column].nunique(
        dropna=False
    )
    <= 1
]

metadata_predictor_overlap = sorted(
    (
        set(IDENTIFIER_METADATA_COLUMNS)
        | set(SPLIT_METADATA_COLUMNS)
        | set(AUDIT_ONLY_COLUMNS)
        | set(TARGET_COLUMNS)
    )
    & set(PREDICTOR_COLUMNS)
)

forbidden_predictor_overlap = sorted(
    set(PREDICTOR_COLUMNS)
    & FORBIDDEN_MODEL_COLUMNS
)

forbidden_source_columns_present = sorted(
    excluded_source_columns.intersection(
        analytical_table.columns
    )
)

NONNEGATIVE_FEATURE_COLUMNS = [
    "rusher_speed",
    "rusher_acceleration",
    "rusher_displacement",
    "quarterback_speed",
    "quarterback_acceleration",
    "quarterback_displacement",
    "rusher_qb_absolute_lateral_gap",
    "rusher_qb_distance",
    "nearest_blocker_speed",
    "nearest_blocker_acceleration",
    "nearest_blocker_displacement",
    "nearest_blocker_absolute_lateral_offset",
    "nearest_blocker_distance",
    "play_pass_rusher_count",
    "play_pass_blocker_count",
]

nonnegative_violations = int(
    analytical_table[
        NONNEGATIVE_FEATURE_COLUMNS
    ].lt(0.0).sum().sum()
)

alignment_range_violations = int(
    (
        analytical_table[
            ALIGNMENT_FEATURE_COLUMNS
        ].lt(-1.0 - 1e-12)
        | analytical_table[
            ALIGNMENT_FEATURE_COLUMNS
        ].gt(1.0 + 1e-12)
    ).sum().sum()
)

cyclical_range_violations = int(
    (
        analytical_table[
            CYCLICAL_ANGLE_FEATURE_COLUMNS
        ].lt(-1.0 - 1e-12)
        | analytical_table[
            CYCLICAL_ANGLE_FEATURE_COLUMNS
        ].gt(1.0 + 1e-12)
    ).sum().sum()
)

target_values = sorted(
    analytical_table["pressure"].unique().tolist()
)

assert len(analytical_table) == 36259
assert analytical_table.shape[1] == 70
assert not analytical_table.duplicated(RUSHER_KEYS).any()
assert analytical_table[PLAY_KEYS].drop_duplicates().shape[0] == 8531
assert sorted(
    analytical_table["actual_week"].unique()
) == list(range(1, 9))
assert analytical_table.isna().sum().sum() == 0
assert np.isfinite(numeric_predictor_values).all()
assert target_values == [0, 1]
assert int(analytical_table["pressure"].sum()) == 4214
assert len(column_inventory) == 70
assert len(PREDICTOR_COLUMNS) == 51
assert len(NUMERIC_PREDICTOR_COLUMNS) == 50
assert len(CATEGORICAL_PREDICTOR_COLUMNS) == 1
assert not metadata_predictor_overlap
assert not forbidden_predictor_overlap
assert not forbidden_source_columns_present
assert not constant_predictor_columns
assert nonnegative_violations == 0
assert alignment_range_violations == 0
assert cyclical_range_violations == 0
assert "football_y" not in analytical_table.columns
assert "football_y_norm" not in analytical_table.columns
assert "has_valid_manual_snap" not in PREDICTOR_COLUMNS

final_integrity_audit = pd.DataFrame(
    [
        {
            "check_id": "AT-01",
            "area": "Analytical grain",
            "status": "PASS",
            "result": (
                "36,259 rows; 36,259 unique rusher keys; "
                "8,531 plays"
            ),
        },
        {
            "check_id": "AT-02",
            "area": "Target source parity",
            "status": "PASS",
            "result": (
                "Exact key, position, and target match "
                "against Stage 3 Parquet"
            ),
        },
        {
            "check_id": "AT-03",
            "area": "Temporal coverage",
            "status": "PASS",
            "result": (
                "Actual weeks 1-8; weekly rows and positives "
                "match source labels"
            ),
        },
        {
            "check_id": "AT-04",
            "area": "Missing and finite values",
            "status": "PASS",
            "result": "0 missing values; 0 non-finite predictors",
        },
        {
            "check_id": "AT-05",
            "area": "Target integrity",
            "status": "PASS",
            "result": (
                "Binary target; 4,214 positives; "
                "11.622% prevalence"
            ),
        },
        {
            "check_id": "AT-06",
            "area": "Column inventory",
            "status": "PASS",
            "result": (
                "70 columns assigned exactly once "
                "to a documented role"
            ),
        },
        {
            "check_id": "AT-07",
            "area": "Predictor schema",
            "status": "PASS",
            "result": (
                "51 eligible predictors: "
                "50 numeric and 1 categorical"
            ),
        },
        {
            "check_id": "AT-08",
            "area": "Leakage exclusion",
            "status": "PASS",
            "result": (
                "No target components, outcome scouting "
                "columns, or QC flags in predictors"
            ),
        },
        {
            "check_id": "AT-09",
            "area": "Metadata separation",
            "status": "PASS",
            "result": (
                "Identifiers, actual_week, audit fields, "
                "and target excluded from predictors"
            ),
        },
        {
            "check_id": "AT-10",
            "area": "Feature ranges",
            "status": "PASS",
            "result": (
                "Distances and magnitudes nonnegative; "
                "cyclical and alignment features valid"
            ),
        },
        {
            "check_id": "AT-11",
            "area": "Predictor variability",
            "status": "PASS",
            "result": "No constant predictor columns",
        },
        {
            "check_id": "AT-12",
            "area": "Football anomaly isolation",
            "status": "PASS",
            "result": (
                "football_y excluded; football_x retained "
                "as longitudinal snap reference"
            ),
        },
    ]
)

assert final_integrity_audit["status"].eq(
    "PASS"
).all()

print(final_integrity_audit)

   check_id                        area status  \
0     AT-01            Analytical grain   PASS   
1     AT-02        Target source parity   PASS   
2     AT-03           Temporal coverage   PASS   
3     AT-04   Missing and finite values   PASS   
4     AT-05            Target integrity   PASS   
5     AT-06            Column inventory   PASS   
6     AT-07            Predictor schema   PASS   
7     AT-08           Leakage exclusion   PASS   
8     AT-09         Metadata separation   PASS   
9     AT-10              Feature ranges   PASS   
10    AT-11       Predictor variability   PASS   
11    AT-12  Football anomaly isolation   PASS   

                                               result  
0   36,259 rows; 36,259 unique rusher keys; 8,531 ...  
1   Exact key, position, and target match against ...  
2   Actual weeks 1-8; weekly rows and positives ma...  
3           0 missing values; 0 non-finite predictors  
4   Binary target; 4,214 positives; 11.622% preval...  
5   70 column

## Final analytical-table contract

### Unit of analysis

Each row represents one defender assigned the `Pass Rush` role in one
validated modeling play, observed at the manually validated snap frame.

Final analytical grain:

- 36,259 pass-rusher observations.
- 36,259 unique `(actual_week, gameId, playId, pass_rusher_nfl_id)` keys.
- 8,531 modeling plays.
- 698 unique pass rushers.
- 4,214 pressure positives.
- Target prevalence: 11.622%.

### Column roles

The analytical table contains 70 columns:

- 5 identifier columns used only for joins, grouping, and traceability.
- 1 temporal split column: `actual_week`.
- 12 audit-only columns.
- 1 binary target: `pressure`.
- 1 categorical predictor.
- 50 numeric predictors.

The future model matrix contains 51 eligible predictors in total.

### Feature families

The predictor set contains:

1. Normalized longitudinal and lateral spatial relationships.
2. Snap-time speed, acceleration, and displacement.
3. Pass-rusher distance and position relative to the quarterback and ball.
4. Geometric relationship with the nearest `Pass Block` player.
5. Number of pass rushers and pass blockers in the play.
6. Cyclical sine/cosine representations of player angles.
7. Velocity components, target alignment, and relative closing speeds.
8. One categorical pre-snap alignment field:
   `rusher_position_lined_up`.

### Modeling contract

- `X`: only columns classified as `categorical_predictor` or
  `numeric_predictor`.
- `y`: only `pressure`.
- `actual_week`: temporal split metadata only.
- `gameId`, `playId`, and all player identifiers: traceability and grouping
  only.
- Raw angles: audit only; their sine/cosine representations must be used
  instead.
- Original `playDirection`: audit only because coordinates were normalized.
- `has_valid_manual_snap`: quality-control field only and not a predictor.
- `pff_hit`, `pff_hurry`, `pff_sack`, and all other outcome-derived PFF
  fields are prohibited from the predictor matrix.

### Methodological caveats

- The nearest blocker is a geometric snap-time proxy and not a confirmed
  blocking assignment.
- `football_x` is treated as a longitudinal snap reference, not as an
  infallible line-of-scrimmage measurement.
- `football_y` is excluded because three validated plays contain boundary
  anomalies.
- Signed spatial exceptions were retained without clipping or subjective
  replacement.
- No analytical row was removed according to its target value.
- All model predictors contain zero missing and zero non-finite values.

### Final integrity status

All analytical-table checks `AT-01` through `AT-12` passed.

In [48]:
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)
REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ANALYTICAL_TABLE_PATH = (
    PROCESSED_DATA_DIR
    / "pass_rush_analytical_table.parquet"
)
COLUMN_INVENTORY_PATH = (
    REPORTS_DIR
    / "analytical_table_column_inventory.csv"
)
ANALYTICAL_QUALITY_REPORT_PATH = (
    REPORTS_DIR
    / "analytical_table_quality_report.csv"
)

EXPORT_COLUMN_ORDER = [
    *SPLIT_METADATA_COLUMNS,
    *IDENTIFIER_METADATA_COLUMNS,
    *AUDIT_ONLY_COLUMNS,
    *CATEGORICAL_PREDICTOR_COLUMNS,
    *NUMERIC_PREDICTOR_COLUMNS,
    *TARGET_COLUMNS,
]

assert len(EXPORT_COLUMN_ORDER) == 70
assert len(set(EXPORT_COLUMN_ORDER)) == 70
assert set(EXPORT_COLUMN_ORDER) == set(
    analytical_table.columns
)

analytical_table_export = (
    analytical_table[
        EXPORT_COLUMN_ORDER
    ]
    .sort_values(RUSHER_KEYS)
    .reset_index(drop=True)
)

assert len(analytical_table_export) == 36259
assert analytical_table_export.shape[1] == 70
assert not analytical_table_export.duplicated(
    RUSHER_KEYS
).any()
assert analytical_table_export.isna().sum().sum() == 0
assert int(
    analytical_table_export["pressure"].sum()
) == 4214

column_position_map = {
    column: position
    for position, column in enumerate(
        EXPORT_COLUMN_ORDER,
        start=1,
    )
}

column_inventory_export = (
    column_inventory.assign(
        column_position=column_inventory[
            "column"
        ].map(column_position_map)
    )
    [
        [
            "column_position",
            "column",
            "column_role",
            "dtype",
            "missing",
            "unique_values",
            "model_eligible",
        ]
    ]
    .sort_values("column_position")
    .reset_index(drop=True)
)

assert len(column_inventory_export) == 70
assert column_inventory_export[
    "column_position"
].tolist() == list(range(1, 71))

quality_report_export = (
    final_integrity_audit.copy()
    .sort_values("check_id")
    .reset_index(drop=True)
)

assert len(quality_report_export) == 12
assert quality_report_export["status"].eq(
    "PASS"
).all()

analytical_table_export.to_parquet(
    ANALYTICAL_TABLE_PATH,
    index=False,
    engine="pyarrow",
    compression="snappy",
)

column_inventory_export.to_csv(
    COLUMN_INVENTORY_PATH,
    index=False,
    encoding="utf-8",
)

quality_report_export.to_csv(
    ANALYTICAL_QUALITY_REPORT_PATH,
    index=False,
    encoding="utf-8",
)

artifact_specs = [
    {
        "artifact": "analytical_table",
        "path": ANALYTICAL_TABLE_PATH,
        "rows": len(analytical_table_export),
        "columns": analytical_table_export.shape[1],
    },
    {
        "artifact": "column_inventory",
        "path": COLUMN_INVENTORY_PATH,
        "rows": len(column_inventory_export),
        "columns": column_inventory_export.shape[1],
    },
    {
        "artifact": "quality_report",
        "path": ANALYTICAL_QUALITY_REPORT_PATH,
        "rows": len(quality_report_export),
        "columns": quality_report_export.shape[1],
    },
]

for artifact_spec in artifact_specs:
    assert artifact_spec["path"].exists()
    assert artifact_spec["path"].stat().st_size > 0

export_artifact_manifest = pd.DataFrame(
    [
        {
            "artifact": artifact_spec["artifact"],
            "relative_path": artifact_spec[
                "path"
            ].relative_to(PROJECT_ROOT).as_posix(),
            "rows": artifact_spec["rows"],
            "columns": artifact_spec["columns"],
            "size_mib": round(
                artifact_spec["path"].stat().st_size
                / (1024 ** 2),
                3,
            ),
        }
        for artifact_spec in artifact_specs
    ]
)

export_artifact_manifest

,artifact,relative_path,rows,columns,size_mib
0,analytical_table,data/processed/pass_rush_analytical_table.parquet,36259,70,8.504
1,column_inventory,reports/analytical_table_column_inventory.csv,70,7,0.004
2,quality_report,reports/analytical_table_quality_report.csv,12,4,0.001


In [50]:
import hashlib


def calculate_sha256(file_path):
    """Calculate a file's SHA-256 checksum."""
    sha256_hash = hashlib.sha256()

    with file_path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            sha256_hash.update(chunk)

    return sha256_hash.hexdigest()


analytical_table_readback = pd.read_parquet(
    ANALYTICAL_TABLE_PATH,
    engine="pyarrow",
)

column_inventory_readback = pd.read_csv(
    COLUMN_INVENTORY_PATH,
    encoding="utf-8",
)

quality_report_readback = pd.read_csv(
    ANALYTICAL_QUALITY_REPORT_PATH,
    encoding="utf-8",
)

pd.testing.assert_frame_equal(
    analytical_table_export,
    analytical_table_readback,
    check_dtype=True,
    check_exact=True,
)

pd.testing.assert_frame_equal(
    column_inventory_export,
    column_inventory_readback,
    check_dtype=True,
    check_exact=True,
)

pd.testing.assert_frame_equal(
    quality_report_export,
    quality_report_readback,
    check_dtype=True,
    check_exact=True,
)

assert analytical_table_readback.shape == (
    36259,
    70,
)
assert not analytical_table_readback.duplicated(
    RUSHER_KEYS
).any()
assert analytical_table_readback.isna().sum().sum() == 0
assert int(
    analytical_table_readback["pressure"].sum()
) == 4214
assert analytical_table_readback[
    PLAY_KEYS
].drop_duplicates().shape[0] == 8531

assert len(column_inventory_readback) == 70
assert int(
    column_inventory_readback[
        "model_eligible"
    ].sum()
) == 51
assert column_inventory_readback[
    "missing"
].sum() == 0

assert len(quality_report_readback) == 12
assert quality_report_readback[
    "status"
].eq("PASS").all()

assert not any(
    column.startswith("Unnamed:")
    for column in column_inventory_readback.columns
)
assert not any(
    column.startswith("Unnamed:")
    for column in quality_report_readback.columns
)

readback_artifacts = [
    {
        "artifact": "analytical_table",
        "path": ANALYTICAL_TABLE_PATH,
        "rows": len(analytical_table_readback),
        "columns": analytical_table_readback.shape[1],
    },
    {
        "artifact": "column_inventory",
        "path": COLUMN_INVENTORY_PATH,
        "rows": len(column_inventory_readback),
        "columns": column_inventory_readback.shape[1],
    },
    {
        "artifact": "quality_report",
        "path": ANALYTICAL_QUALITY_REPORT_PATH,
        "rows": len(quality_report_readback),
        "columns": quality_report_readback.shape[1],
    },
]

readback_validation_summary = pd.DataFrame(
    [
        {
            "artifact": artifact["artifact"],
            "rows": artifact["rows"],
            "columns": artifact["columns"],
            "exact_match": True,
            "sha256": calculate_sha256(
                artifact["path"]
            ),
            "status": "PASS",
        }
        for artifact in readback_artifacts
    ]
)

assert readback_validation_summary[
    "status"
].eq("PASS").all()
assert readback_validation_summary[
    "exact_match"
].all()

print(readback_validation_summary)

           artifact   rows  columns  exact_match  \
0  analytical_table  36259       70         True   
1  column_inventory     70        7         True   
2    quality_report     12        4         True   

                                              sha256 status  
0  b7198bf0f347a8bf17f9d468f801f61bd3ee4400a28dd1...   PASS  
1  75f966d3207164bb95a86d2487250515bb1fbed67f9b8a...   PASS  
2  fd51b902466192ea22a793a324cb40e6c88f0b2dad42fe...   PASS  
